# ZeroCap GPT-2 Base — Colab T4 prototype

Notebook này chứa toàn bộ prototype ZeroCap end-to-end để chạy tuần tự bằng Google Colab GPU T4. Đây là matched-backbone adaptation dùng GPT-2 Base để so sánh với ClipCap, không phải exact GPT-2 Medium reproduction của paper. Baseline chính dùng final CLIP-only reranking. Mặc định chỉ chạy smoke mode trên một ảnh VAL cố định; không huấn luyện, không đọc reference caption trong generation, và không tự chạy toàn bộ TEST.

Quy trình dự kiến: mở từ GitHub → chọn Runtime GPU T4 → đặt RUN_MODE → Run all. `val_tune` là mode chẩn đoán riêng, chạy các case one-factor-at-a-time trên cùng fixed VAL IDs và tuyệt đối không thay đổi model weights hay benchmark chuẩn. Revision hiện tại thêm GPT-2 BOS đúng baseline, bỏ BOS trước CLIP guidance và dùng caption-only cho final CLIP rerank. Logic được chia thành class/function độc lập để có thể trích xuất nguyên trạng sang các module Python sau khi prototype đã PASS trên Colab.

In [ ]:
REPO_URL = "https://github.com/HnhanBk415/zfs-clip-image-captioning.git"
BRANCH = "feat/hoangnhan/test_zerocap"
PROJECT_DIR = "/content/zfs-clip-image-captioning"

RUN_MODE = "smoke"

MOUNT_DRIVE_FOR_OUTPUTS = True
DRIVE_OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "zfs-clip-image-captioning/outputs/zerocap"
)

TIME_BUDGET_HOURS = 4.0
BENCHMARK_VAL_IMAGES = 5
RUN_HEAVY_METRICS = False

## 1. Clone/update repo an toàn

Cell này không clone đè. Nếu thư mục đã tồn tại, nó yêu cầu đúng remote, đúng branch, worktree sạch, rồi chỉ fast-forward. Nếu validation thất bại, hãy Restart session trước khi Run all lại.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def run_git(arguments, cwd=None):
    completed = subprocess.run(
        ["git", *arguments],
        cwd=cwd,
        check=True,
        text=True,
        capture_output=True,
    )
    return completed.stdout.strip()


project_path = Path(PROJECT_DIR)
if not project_path.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            REPO_URL,
            str(project_path),
        ],
        check=True,
    )
else:
    if not (project_path / ".git").is_dir():
        raise RuntimeError(
            f"{project_path} already exists but is not a Git repository. "
            "Restart the Colab session or choose an empty PROJECT_DIR."
        )

    actual_origin = run_git(
        ["config", "--get", "remote.origin.url"],
        cwd=project_path,
    )
    normalize_remote = lambda value: value.rstrip("/").removesuffix(".git").lower()
    if normalize_remote(actual_origin) != normalize_remote(REPO_URL):
        raise RuntimeError(
            "Existing PROJECT_DIR points to a different origin. "
            "Restart the Colab session before Run all."
        )

    dirty = run_git(["status", "--porcelain"], cwd=project_path)
    if dirty:
        raise RuntimeError(
            "Existing Colab clone has local changes. Nothing was overwritten. "
            "Restart the session, or commit/archive those changes manually."
        )

    current_branch = run_git(
        ["branch", "--show-current"],
        cwd=project_path,
    )
    if current_branch != BRANCH:
        raise RuntimeError(
            f"Existing clone is on {current_branch!r}, expected {BRANCH!r}. "
            "Restart the Colab session before Run all."
        )

    subprocess.run(
        ["git", "fetch", "origin", BRANCH],
        cwd=project_path,
        check=True,
    )
    subprocess.run(
        ["git", "merge", "--ff-only", f"origin/{BRANCH}"],
        cwd=project_path,
        check=True,
    )

os.chdir(project_path)
if str(project_path) not in sys.path:
    sys.path.insert(0, str(project_path))

GIT_COMMIT = run_git(["rev-parse", "HEAD"], cwd=project_path)
print("Project root:", project_path)
print("Branch:", BRANCH)
print("Git commit:", GIT_COMMIT)

## 2. Pinned Colab dependencies

Transformers và tokenizer được pin để giữ legacy tuple past_key_values ổn định. PyTorch CUDA của Colab được giữ nguyên để không phá binary/runtime GPU do Colab cung cấp.

In [ ]:
%pip install --quiet --upgrade \
    "transformers==4.56.2" \
    "tokenizers==0.22.1" \
    "huggingface-hub==0.34.4" \
    "safetensors==0.4.5" \
    "kagglehub==0.3.12" \
    "pycocoevalcap==1.2" \
    "nltk==3.9.1" \
    "rouge-score==0.1.2"

In [ ]:
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import platform
import random
import signal
import shutil
import statistics
import time
import traceback
import urllib.request
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    CLIPModel,
    CLIPProcessor,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
)

if transformers.__version__ != "4.56.2":
    raise RuntimeError(
        "This notebook requires transformers==4.56.2. "
        f"Loaded {transformers.__version__}; restart the Colab session and Run all."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In Colab choose Runtime → Change runtime type "
        "→ T4 GPU, then Restart session and Run all."
    )

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
LIBRARY_VERSIONS = {
    name: importlib.metadata.version(name)
    for name in [
        "torch",
        "transformers",
        "tokenizers",
        "huggingface-hub",
        "kagglehub",
        "numpy",
        "Pillow",
        "pandas",
        "pycocoevalcap",
        "nltk",
    ]
}

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", GPU_NAME)
print("Git commit:", GIT_COMMIT)

## 3. Single source of truth: ZeroCapRunConfig

Mọi tham số thuật toán/profile chỉ được khai báo trong cell này. Smoke dùng đúng debug profile; benchmark, val_tune, test_smoke và final_test dùng final profile. `val_tune` tạo identity và run directory riêng cho từng candidate để không resume chéo. Config hash bao gồm commit và digest split manifest để một revision/input mới luôn tạo run directory mới.

In [ ]:
@dataclass(frozen=True)
class ZeroCapRunConfig:
    run_mode: str

    gpt_model: str = "openai-community/gpt2"
    clip_model: str = "openai/clip-vit-base-patch32"
    device: str = "cuda"
    model_dtype: str = "float32"
    algorithm_revision: str = "matched_backbone_base_v2_bos_caption_rerank"
    val_tune_candidate: str = ""
    val_tune_case_reason: str = ""

    prompt: str = "Image of a"
    prepend_bos_token: bool = True
    top_k: int = 512
    inner_iterations: int = 5
    beam_size: int = 5
    max_new_tokens: int = 15

    clip_temperature: float = 0.01
    clip_loss_scale: float = 1.0
    fluency_weight: float = 0.2

    step_size: float = 0.3
    grad_norm_factor: float = 0.9
    fusion_factor: float = 0.99

    reset_context_delta: bool = True
    repetition_penalty: float = 1.0
    min_new_tokens: int = 2

    stop_token: str = "."
    end_factor: float = 1.01
    forbidden_factor: float = 20.0

    gradient_eps: float = 1e-15
    probability_eps: float = 1e-12
    clip_candidate_chunk_size: int = 64
    clip_max_text_tokens: int = 77
    beam_length_penalty: float = 1.0
    final_rerank_mode: str = "clip_only"
    final_rerank_text_mode: str = "caption_only"
    final_rerank_clip_weight: float = 1.0
    final_rerank_beam_weight: float = 1.0
    seed: int = 42

    save_diagnostics: bool = False
    diagnostic_top_n: int = 20
    diagnostic_save_all_top_k: bool = False
    save_diagnostic_images: bool = False
    display_input_images: bool = False

    benchmark_warmup_images: int = 2
    benchmark_val_images: int = 5
    test_smoke_images: int = 5
    time_budget_hours: float = 4.0
    run_heavy_metrics: bool = False
    metric_timeout_sec: float = 120.0

    feature_cache_path: Optional[str] = None
    forbidden_tokens_url: str = (
        "https://raw.githubusercontent.com/"
        "YoadTew/zero-shot-image-to-text/main/forbidden_tokens.npy"
    )

    debug: bool = False
    git_commit: str = ""
    split_manifest_sha256: str = ""
    output_root: str = ""
    config_hash: str = ""
    run_dir: str = ""

    @classmethod
    def _with_identity(cls, draft):
        if draft.fluency_weight < 0:
            raise ValueError("fluency_weight must be non-negative.")
        if not 0 < draft.fusion_factor <= 1:
            raise ValueError("fusion_factor must be in (0, 1].")
        if draft.clip_temperature <= 0:
            raise ValueError("clip_temperature must be positive.")
        if draft.top_k <= 0 or draft.inner_iterations <= 0:
            raise ValueError("top_k and inner_iterations must be positive.")
        if draft.beam_size <= 0 or draft.max_new_tokens <= 0:
            raise ValueError("beam_size and max_new_tokens must be positive.")
        if draft.step_size <= 0 or draft.grad_norm_factor <= 0:
            raise ValueError("step_size and grad_norm_factor must be positive.")
        hash_payload = asdict(draft)
        for derived_key in ("output_root", "config_hash", "run_dir"):
            hash_payload.pop(derived_key)
        canonical = json.dumps(
            hash_payload,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
        config_hash = hashlib.sha256(canonical).hexdigest()[:12]
        run_dir = Path(draft.output_root) / (
            f"{draft.run_mode}_{config_hash}"
        )
        return replace(
            draft,
            config_hash=config_hash,
            run_dir=str(run_dir),
        )

    @staticmethod
    def val_tune_grid():
        return (
            (
                "A_baseline",
                {
                    "fluency_weight": 0.2,
                    "fusion_factor": 0.99,
                    "clip_temperature": 0.01,
                    "case_reason": (
                        "Official ZeroCap inference defaults with the "
                        "matched GPT-2 Base backbone."
                    ),
                },
            ),
            (
                "B_fluency_0_10",
                {
                    "fluency_weight": 0.1,
                    "fusion_factor": 0.99,
                    "clip_temperature": 0.01,
                    "case_reason": (
                        "Reduce the KL/fluency constraint so CLIP can steer "
                        "the context more strongly; grammar may weaken."
                    ),
                },
            ),
            (
                "C_fluency_0_05",
                {
                    "fluency_weight": 0.05,
                    "fusion_factor": 0.99,
                    "clip_temperature": 0.01,
                    "case_reason": (
                        "Stress-test stronger visual steering with a much "
                        "weaker GPT-2 KL constraint."
                    ),
                },
            ),
            (
                "D_clip_temperature_0_02",
                {
                    "fluency_weight": 0.2,
                    "fusion_factor": 0.99,
                    "clip_temperature": 0.02,
                    "case_reason": (
                        "Soften the CLIP target distribution to reduce "
                        "overreaction to small/noisy similarity gaps."
                    ),
                },
            ),
            (
                "E_fusion_0_95",
                {
                    "fluency_weight": 0.2,
                    "fusion_factor": 0.95,
                    "clip_temperature": 0.01,
                    "case_reason": (
                        "Give the untouched GPT-2 distribution more weight "
                        "only at final token fusion."
                    ),
                },
            ),
        )

    def for_val_tune_candidate(self, candidate_name, overrides):
        if self.run_mode != "val_tune":
            raise ValueError(
                "VAL-tune candidates require RUN_MODE='val_tune'."
            )
        allowed_overrides = {
            "fluency_weight",
            "fusion_factor",
            "clip_temperature",
            "case_reason",
        }
        unexpected = set(overrides) - allowed_overrides
        if unexpected:
            raise ValueError(
                f"Unsupported VAL-tune overrides: {sorted(unexpected)}"
            )
        if set(overrides) != allowed_overrides:
            raise ValueError(
                "Every candidate must set fluency_weight, fusion_factor, "
                "clip_temperature and case_reason."
            )
        candidate_root = Path(self.run_dir) / "candidates"
        draft = replace(
            self,
            val_tune_candidate=str(candidate_name),
            val_tune_case_reason=str(overrides["case_reason"]),
            fluency_weight=float(overrides["fluency_weight"]),
            fusion_factor=float(overrides["fusion_factor"]),
            clip_temperature=float(overrides["clip_temperature"]),
            save_diagnostics=True,
            save_diagnostic_images=False,
            display_input_images=False,
            debug=False,
            output_root=str(candidate_root),
            config_hash="",
            run_dir="",
        )
        return type(self)._with_identity(draft)

    @classmethod
    def for_mode(
        cls,
        run_mode,
        git_commit,
        split_manifest_sha256,
        output_root,
        time_budget_hours,
        benchmark_val_images,
        run_heavy_metrics,
    ):
        allowed_modes = {
            "smoke",
            "benchmark",
            "val_tune",
            "test_smoke",
            "final_test",
        }
        if run_mode not in allowed_modes:
            raise ValueError(
                f"RUN_MODE must be one of {sorted(allowed_modes)}, got {run_mode!r}."
            )
        if benchmark_val_images not in {5, 10}:
            raise ValueError("BENCHMARK_VAL_IMAGES must be 5 or 10.")

        draft = cls(
            run_mode=run_mode,
            git_commit=git_commit,
            split_manifest_sha256=split_manifest_sha256,
            output_root=str(output_root),
            time_budget_hours=float(time_budget_hours),
            benchmark_val_images=int(benchmark_val_images),
            run_heavy_metrics=bool(run_heavy_metrics),
        )
        allowed_rerank_modes = {"clip_only", "clip_beam"}
        if draft.final_rerank_mode not in allowed_rerank_modes:
            raise ValueError(
                "final_rerank_mode must be one of "
                f"{sorted(allowed_rerank_modes)}."
            )
        if (
            draft.final_rerank_clip_weight < 0
            or draft.final_rerank_beam_weight < 0
        ):
            raise ValueError("Final rerank weights must be non-negative.")
        if (
            draft.final_rerank_clip_weight == 0
            and draft.final_rerank_beam_weight == 0
        ):
            raise ValueError("At least one final rerank weight must be positive.")
        if (
            draft.final_rerank_mode == "clip_only"
            and draft.final_rerank_clip_weight == 0
        ):
            raise ValueError("clip_only reranking requires a positive CLIP weight.")
        allowed_rerank_text_modes = {"caption_only", "prompt_plus_caption"}
        if draft.final_rerank_text_mode not in allowed_rerank_text_modes:
            raise ValueError(
                "final_rerank_text_mode must be one of "
                f"{sorted(allowed_rerank_text_modes)}."
            )
        if not draft.prepend_bos_token:
            raise ValueError(
                "The baseline-faithful prototype requires prepend_bos_token=True."
            )
        if draft.metric_timeout_sec <= 0:
            raise ValueError("metric_timeout_sec must be positive.")

        if run_mode == "smoke":
            draft = replace(
                draft,
                top_k=5,
                inner_iterations=1,
                beam_size=1,
                max_new_tokens=3,
                clip_candidate_chunk_size=5,
                save_diagnostics=True,
                save_diagnostic_images=True,
                display_input_images=True,
                debug=True,
            )

        return cls._with_identity(draft)

    def public_dict(self):
        return asdict(self)


split_manifest_path = (
    project_path / "data" / "flickr8k" / "metadata" / "split_manifest.json"
)
if not split_manifest_path.is_file():
    raise FileNotFoundError(split_manifest_path)

SPLIT_MANIFEST_SHA256 = hashlib.sha256(
    split_manifest_path.read_bytes()
).hexdigest()

if MOUNT_DRIVE_FOR_OUTPUTS:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_ROOT = Path(DRIVE_OUTPUT_DIR)
else:
    OUTPUT_ROOT = project_path / "outputs" / "zerocap"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

config = ZeroCapRunConfig.for_mode(
    run_mode=RUN_MODE,
    git_commit=GIT_COMMIT,
    split_manifest_sha256=SPLIT_MANIFEST_SHA256,
    output_root=OUTPUT_ROOT,
    time_budget_hours=TIME_BUDGET_HOURS,
    benchmark_val_images=BENCHMARK_VAL_IMAGES,
    run_heavy_metrics=RUN_HEAVY_METRICS,
)


def seed_everything(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(config.seed)
RUN_DIR = Path(config.run_dir)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Run mode:", config.run_mode)
print("Profile:", "debug" if config.debug else "final")
print("Algorithm revision:", config.algorithm_revision)
print("Config hash:", config.config_hash)
print("Global random seed:", config.seed)
print("Run directory:", RUN_DIR)
if config.save_diagnostics:
    print(
        "DIAGNOSTIC TRACE ENABLED: benchmark timing includes trace overhead; "
        "use it for diagnosis, not final TEST sizing."
    )

## 4. Explicit data access boundary

Generation lấy image IDs từ split_manifest.json, là file chỉ chứa ID. Chỉ Flickr8kData.load_references mới được mở val.json/test.json, và method đó chỉ được evaluator gọi sau khi prediction hoàn tất.

In [ ]:
class Flickr8kData:
    def __init__(self, project_root):
        self.project_root = Path(project_root)
        self.data_root = self.project_root / "data" / "flickr8k"
        self.manifest_path = self.data_root / "metadata" / "split_manifest.json"
        with self.manifest_path.open("r", encoding="utf-8") as handle:
            self.manifest = json.load(handle)
        self.references_loaded = False
        self.images_dir = None
        self.image_paths = {}

    def split_ids(self, split):
        if split not in {"train", "val", "test"}:
            raise ValueError(f"Unsupported split: {split}")
        values = self.manifest["image_ids"][split]
        if len(values) != len(set(values)):
            raise AssertionError(f"Duplicate IDs in {split} split.")
        return list(values)

    def download_and_index_images(self):
        tracked_raw_dir = self.data_root / "raw" / "Images"
        if tracked_raw_dir.is_dir() and any(tracked_raw_dir.glob("*.jpg")):
            download_root = tracked_raw_dir.parent
            print("Using existing raw image directory:", tracked_raw_dir)
        else:
            try:
                from src.config.common_config import KAGGLE_DATASET_HANDLE
                print(
                    "Kaggle dataset handle imported from common config:",
                    KAGGLE_DATASET_HANDLE,
                )
            except Exception as exc:
                KAGGLE_DATASET_HANDLE = "adityajn105/flickr8k"
                print(
                    "Common config import unavailable; using verified fallback "
                    f"{KAGGLE_DATASET_HANDLE!r}. Cause: {type(exc).__name__}"
                )

            try:
                download_root = Path(
                    kagglehub.dataset_download(KAGGLE_DATASET_HANDLE)
                )
            except Exception as exc:
                raise RuntimeError(
                    "KaggleHub could not download Flickr8k. Check Colab network/"
                    "Kaggle access, then restart and Run all."
                ) from exc

        candidates = [
            path
            for path in Path(download_root).rglob("Images")
            if path.is_dir()
        ]
        if tracked_raw_dir.is_dir():
            candidates.append(tracked_raw_dir)

        candidates = list(dict.fromkeys(path.resolve() for path in candidates))
        if not candidates:
            raise FileNotFoundError(
                f"No directory named Images was found under {download_root}."
            )

        ranked = []
        for candidate in candidates:
            image_files = [
                path
                for path in candidate.rglob("*")
                if path.is_file()
                and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
            ]
            ranked.append((len(image_files), candidate, image_files))

        count, selected_dir, image_files = max(ranked, key=lambda item: item[0])
        if count == 0:
            raise RuntimeError(f"Images directory is empty: {selected_dir}")

        paths = {}
        for path in image_files:
            if path.name in paths and paths[path.name] != path:
                raise RuntimeError(f"Duplicate image filename: {path.name}")
            paths[path.name] = path

        self.images_dir = selected_dir
        self.image_paths = paths
        print("Flickr8k Images directory:", selected_dir)
        print("Indexed raw images:", len(paths))
        return selected_dir

    def assert_ids_exist(self, image_ids):
        missing = [image_id for image_id in image_ids if image_id not in self.image_paths]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} selected image IDs are missing; first: {missing[:5]}"
            )

    def load_image(self, image_id):
        if image_id not in self.image_paths:
            raise KeyError(f"Image ID was not indexed: {image_id}")
        with Image.open(self.image_paths[image_id]) as opened:
            return opened.convert("RGB")

    def load_references(self, split, image_ids):
        if split not in {"val", "test"}:
            raise ValueError("References are only supported for val/test evaluation.")
        split_path = self.data_root / "splits" / f"{split}.json"
        with split_path.open("r", encoding="utf-8") as handle:
            references_by_id = json.load(handle)
        self.references_loaded = True

        selected = {}
        for image_id in image_ids:
            references = references_by_id.get(image_id)
            if references is None:
                raise KeyError(f"No references found for {image_id}")
            if len(references) != 5 or not all(
                isinstance(value, str) and value.strip()
                for value in references
            ):
                raise AssertionError(
                    f"{image_id} must have exactly five non-empty references."
                )
            selected[image_id] = list(references)
        return selected


def deterministic_sample(values, count, seed):
    ordered = sorted(values)
    if count < 0 or count > len(ordered):
        raise ValueError(
            f"Cannot sample {count} items from a population of {len(ordered)}."
        )
    return random.Random(seed).sample(ordered, count)


data = Flickr8kData(project_path)
VAL_IDS = data.split_ids("val")
TEST_IDS = data.split_ids("test")
data.download_and_index_images()

SMOKE_VAL_IMAGE_ID = deterministic_sample(VAL_IDS, 1, config.seed)[0]
remaining_val_ids = [
    image_id for image_id in sorted(VAL_IDS)
    if image_id != SMOKE_VAL_IMAGE_ID
]
other_val_ids = deterministic_sample(
    remaining_val_ids,
    config.benchmark_warmup_images - 1 + config.benchmark_val_images,
    config.seed + 1,
)
VAL_WARMUP_IDS = [
    SMOKE_VAL_IMAGE_ID,
    *other_val_ids[: config.benchmark_warmup_images - 1],
]
VAL_BENCHMARK_IDS = other_val_ids[
    config.benchmark_warmup_images - 1:
]

selected_ids_to_verify = [SMOKE_VAL_IMAGE_ID]
if config.run_mode in {"benchmark", "val_tune"}:
    selected_ids_to_verify.extend(VAL_WARMUP_IDS)
    selected_ids_to_verify.extend(VAL_BENCHMARK_IDS)

data.assert_ids_exist(selected_ids_to_verify)
assert not data.references_loaded
print("Fixed smoke VAL image:", SMOKE_VAL_IMAGE_ID)

## 5. JSON-safe types and invariant helpers

In [ ]:
ContextDelta = Tuple[Tuple[torch.Tensor, torch.Tensor], ...]


@dataclass
class GenerationResult:
    image_id: str
    caption: str
    generated_token_ids: List[int]
    num_generated_tokens: int
    stop_reason: str
    generation_time_sec: float
    end_to_end_time_sec: float
    final_clip_similarity: float
    gpt_model: str
    clip_model: str
    config_hash: str
    git_commit: str

    def to_dict(self):
        value = asdict(self)
        json.dumps(value)
        return value


@dataclass
class ContextStepResult:
    p_original: torch.Tensor
    p_guided_final: torch.Tensor
    context_delta: ContextDelta
    diagnostics: List[Dict[str, Any]]


@dataclass
class BeamState:
    token_ids: torch.Tensor
    generated_token_ids: Tuple[int, ...] = field(default_factory=tuple)
    accumulated_logprob: float = 0.0
    stopped: bool = False
    stop_reason: str = ""
    context_delta: Optional[ContextDelta] = None

    def normalized_score(self, length_penalty):
        length = max(1, len(self.generated_token_ids))
        return self.accumulated_logprob / (length ** length_penalty)


def probability_diagnostics(probabilities, tokenizer, top_n):
    detached = probabilities.detach().float().cpu()
    if detached.ndim != 2 or detached.shape[0] != 1:
        raise AssertionError("Diagnostic probability tensor must be [1, vocab].")
    values = detached[0]
    count = min(int(top_n), int(values.numel()))
    top_values, top_ids = torch.topk(values, k=count)
    entropy = float(
        -(values * torch.log(values.clamp_min(1e-30))).sum().item()
    )
    rows = []
    for rank, (token_id, probability) in enumerate(
        zip(top_ids.tolist(), top_values.tolist()),
        start=1,
    ):
        rows.append(
            {
                "rank": rank,
                "token_id": int(token_id),
                "token_text": tokenizer.decode(
                    [int(token_id)],
                    skip_special_tokens=False,
                    clean_up_tokenization_spaces=False,
                ),
                "probability": float(probability),
                "log_probability": float(
                    math.log(max(float(probability), 1e-30))
                ),
            }
        )
    return {
        "sum": float(values.sum().item()),
        "min": float(values.min().item()),
        "max": float(values.max().item()),
        "entropy": entropy,
        "effective_vocabulary_size": float(math.exp(entropy)),
        "top_tokens": rows,
    }


def context_delta_diagnostics(delta):
    layer_rows = []
    squared_total = 0.0
    for layer_index, (delta_key, delta_value) in enumerate(delta):
        key_norm = float(delta_key.detach().float().norm().item())
        value_norm = float(delta_value.detach().float().norm().item())
        squared_total += key_norm ** 2 + value_norm ** 2
        layer_rows.append(
            {
                "layer": layer_index,
                "key_norm": key_norm,
                "value_norm": value_norm,
            }
        )
    return {
        "global_norm": float(math.sqrt(squared_total)),
        "layers": layer_rows,
    }


def compact_iteration_diagnostics(diagnostics, top_n, save_all_top_k):
    similarities = diagnostics["similarities"]
    candidate_count = len(similarities)
    top_n = min(int(top_n), candidate_count)

    def candidate_row(index):
        return {
            "top_k_index": int(index),
            "token_id": int(diagnostics["top_token_ids"][index]),
            "token_text": diagnostics["decoded_top_tokens"][index],
            "candidate_text": diagnostics["candidate_texts"][index],
            "text_prefix_match": bool(
                diagnostics["candidate_text_prefix_matches"][index]
            ),
            "guided_probability": float(
                diagnostics["top_guided_probabilities"][index]
            ),
            "clip_similarity": float(similarities[index]),
            "clip_target_probability": float(
                diagnostics["target_topk_probabilities"][index]
            ),
        }

    clip_ranked_indices = sorted(
        range(candidate_count),
        key=lambda index: similarities[index],
        reverse=True,
    )[:top_n]
    guided_ranked_indices = list(range(top_n))
    similarity_array = np.asarray(similarities, dtype=np.float64)
    compact = {
        key: diagnostics[key]
        for key in (
            "iteration",
            "evaluation_type",
            "is_best_so_far",
            "loss_clip",
            "loss_fluency",
            "loss_total",
            "p_guided_sum",
            "gradient_norm_min",
            "gradient_norm_max",
            "gradient_norm_mean",
            "target_sum",
            "delta_before",
            "delta_after",
        )
    }
    compact.update(
        {
            "current_text": diagnostics["current_text"],
            "candidate_count": candidate_count,
            "clip_similarity_stats": {
                "min": float(similarity_array.min()),
                "max": float(similarity_array.max()),
                "mean": float(similarity_array.mean()),
                "std": float(similarity_array.std()),
            },
            "top_by_clip_similarity": [
                candidate_row(index) for index in clip_ranked_indices
            ],
            "top_by_guided_probability": [
                candidate_row(index) for index in guided_ranked_indices
            ],
        }
    )
    if save_all_top_k:
        compact["all_top_k_candidates"] = [
            candidate_row(index) for index in range(candidate_count)
        ]
    return compact


class GenerationTrace:
    SCHEMA_VERSION = 2

    def __init__(self, config, image_id, image):
        self.config = config
        self.image_id = str(image_id)
        self.started_at = time.perf_counter()
        self.path = (
            Path(config.run_dir)
            / "diagnostics"
            / f"{Path(self.image_id).name}.json"
        )
        self.image_path = (
            self.path.parent
            / "images"
            / Path(self.image_id).name
        )
        self.payload = {
            "schema_version": self.SCHEMA_VERSION,
            "status": "running",
            "image_id": self.image_id,
            "config_hash": config.config_hash,
            "git_commit": config.git_commit,
            "run_mode": config.run_mode,
            "algorithm_revision": config.algorithm_revision,
            "image": {
                "mode": str(getattr(image, "mode", "unknown")),
                "size": [int(value) for value in getattr(image, "size", ())],
            },
            "trace_settings": {
                "top_n": config.diagnostic_top_n,
                "save_all_top_k": config.diagnostic_save_all_top_k,
            },
            "events": [],
        }
        self._save_input_image(image)

    def _save_input_image(self, image):
        if not self.config.save_diagnostic_images:
            self.payload["image"]["saved_path"] = None
            return
        try:
            self.image_path.parent.mkdir(parents=True, exist_ok=True)
            image.convert("RGB").save(
                self.image_path,
                format="JPEG",
                quality=95,
            )
            self.payload["image"]["saved_path"] = str(
                self.image_path.relative_to(Path(self.config.run_dir))
            )
        except Exception as error:
            self.payload["image"]["saved_path"] = None
            self.payload["image"]["save_error"] = (
                f"{type(error).__name__}: {error}"
            )

    def record(self, event_type, **payload):
        event = {
            "event": str(event_type),
            "elapsed_sec": float(time.perf_counter() - self.started_at),
            **payload,
        }
        json.dumps(event, ensure_ascii=False, allow_nan=False)
        self.payload["events"].append(event)

    def finish(self, result):
        self.payload["status"] = "success"
        self.payload["result"] = dict(result)
        self.payload["total_trace_elapsed_sec"] = float(
            time.perf_counter() - self.started_at
        )

    def fail(self, error):
        self.payload["status"] = "error"
        self.payload["error"] = {
            "type": type(error).__name__,
            "message": str(error),
            "traceback": traceback.format_exc(),
        }
        self.payload["total_trace_elapsed_sec"] = float(
            time.perf_counter() - self.started_at
        )

    def save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        temporary = self.path.with_suffix(self.path.suffix + ".tmp")
        with temporary.open("w", encoding="utf-8") as handle:
            json.dump(
                self.payload,
                handle,
                ensure_ascii=False,
                indent=2,
                allow_nan=False,
            )
        os.replace(temporary, self.path)
        return self.path


def assert_probability_distribution(probabilities, label, tolerance=1e-4):
    if probabilities.ndim != 2 or probabilities.shape[0] != 1:
        raise AssertionError(f"{label} must have shape [1, vocab].")
    if not torch.isfinite(probabilities).all():
        raise AssertionError(f"{label} contains NaN/Inf.")
    if (probabilities < 0).any():
        raise AssertionError(f"{label} contains negative probability.")
    total = probabilities.sum()
    if not torch.allclose(
        total,
        torch.ones_like(total),
        atol=tolerance,
        rtol=tolerance,
    ):
        raise AssertionError(f"{label} sums to {total.item()}, expected 1.")


def parameter_snapshot(model):
    return {
        name: (id(parameter), parameter.data_ptr(), parameter._version)
        for name, parameter in model.named_parameters()
    }


def assert_parameter_snapshot_unchanged(model, before, label):
    after = parameter_snapshot(model)
    if before != after:
        raise AssertionError(f"{label} parameters changed during generation.")


def assert_model_parameter_grads_clear(model, label):
    offenders = [
        name
        for name, parameter in model.named_parameters()
        if parameter.grad is not None
    ]
    if offenders:
        raise AssertionError(
            f"{label} unexpectedly accumulated gradients: {offenders[:5]}"
        )


def clone_context_delta(delta):
    if delta is None:
        return None
    return tuple(
        (
            delta_key.detach().clone(),
            delta_value.detach().clone(),
        )
        for delta_key, delta_value in delta
    )


def postprocess_caption(text, prompt):
    normalized = " ".join(str(text).strip().split())
    if normalized.startswith(prompt):
        normalized = normalized[len(prompt):].strip()
    normalized = normalized.replace(" .", ".")
    return normalized.strip()

## 6. Atomic output and resume manager

Prediction được ghi lại sau từng ảnh. Existing run chỉ được resume khi config_hash và git commit khớp. Vì commit là một phần của run identity hash, code/config revision mới tự tạo directory mới.

In [ ]:
class PredictionStore:
    def __init__(self, config, library_versions, gpu_name):
        self.config = config
        self.run_dir = Path(config.run_dir)
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.metadata_path = self.run_dir / "run_metadata.json"
        self.config_path = self.run_dir / "config.json"
        self.predictions_path = self.run_dir / "predictions.json"
        self.predictions_csv_path = self.run_dir / "predictions.csv"
        self.metrics_path = self.run_dir / "metrics.json"

        expected_metadata = {
            "git_commit": config.git_commit,
            "config_hash": config.config_hash,
            "run_mode": config.run_mode,
            "algorithm_revision": config.algorithm_revision,
            "val_tune_candidate": config.val_tune_candidate,
            "seed": config.seed,
            "gpt_model": config.gpt_model,
            "clip_model": config.clip_model,
            "device": config.device,
            "gpu": gpu_name,
            "library_versions": dict(library_versions),
        }

        if self.metadata_path.exists():
            with self.metadata_path.open("r", encoding="utf-8") as handle:
                existing = json.load(handle)
            for key in ("git_commit", "config_hash", "run_mode"):
                if existing.get(key) != expected_metadata[key]:
                    raise RuntimeError(
                        f"Refusing to resume: metadata {key} mismatch in "
                        f"{self.run_dir}."
                    )
        elif self.predictions_path.exists():
            raise RuntimeError(
                "Predictions exist without run_metadata.json; refusing unsafe resume."
            )
        else:
            self._atomic_json(self.metadata_path, expected_metadata)
            self._atomic_json(self.config_path, config.public_dict())

        self.predictions = {}
        if self.predictions_path.exists():
            with self.predictions_path.open("r", encoding="utf-8") as handle:
                existing_predictions = json.load(handle)
            if not isinstance(existing_predictions, list):
                raise RuntimeError("predictions.json must contain a JSON list.")
            for item in existing_predictions:
                if (
                    item.get("config_hash") != config.config_hash
                    or item.get("git_commit") != config.git_commit
                ):
                    raise RuntimeError(
                        "Refusing to resume a prediction with stale config/commit."
                    )
                self.predictions[item["image_id"]] = item

    @staticmethod
    def _atomic_json(path, payload):
        path = Path(path)
        temporary = path.with_suffix(path.suffix + ".tmp")
        with temporary.open("w", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2)
        os.replace(temporary, path)

    def _write_csv(self):
        rows = list(self.predictions.values())
        if not rows:
            return
        fieldnames = sorted(
            set().union(*(row.keys() for row in rows))
        )
        temporary = self.predictions_csv_path.with_suffix(".csv.tmp")
        with temporary.open(
            "w",
            encoding="utf-8",
            newline="",
        ) as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        os.replace(temporary, self.predictions_csv_path)

    def completed_ids(self):
        return set(self.predictions)

    def save_prediction(self, result):
        json.dumps(result)
        if (
            result.get("config_hash") != self.config.config_hash
            or result.get("git_commit") != self.config.git_commit
        ):
            raise AssertionError("Prediction identity does not match this run.")
        self.predictions[result["image_id"]] = dict(result)
        ordered = list(self.predictions.values())
        self._atomic_json(self.predictions_path, ordered)
        self._write_csv()

    def save_metrics(self, metrics):
        self._atomic_json(self.metrics_path, metrics)

    def save_artifact_json(self, filename, payload):
        path = self.run_dir / filename
        self._atomic_json(path, payload)
        return path


store = PredictionStore(
    config=config,
    library_versions=LIBRARY_VERSIONS,
    gpu_name=GPU_NAME,
)
print("Existing completed predictions:", len(store.completed_ids()))

## 7. Frozen model loader and official forbidden-token artifact

Toàn bộ GPT-2 và CLIP được eval/freeze. Forbidden-token NPY được tải từ repo ZeroCap chính thức, kiểm tra header, dtype/rank/range/uniqueness, đối chiếu GPT-2 vocab và ghi SHA-256 metadata. Download failure là hard failure.

In [ ]:
class ZeroCapModels:
    def __init__(self, config):
        self.config = config
        requested_device = torch.device(config.device)
        self.device = (
            torch.device("cuda", torch.cuda.current_device())
            if requested_device.type == "cuda"
            else requested_device
        )
        self.dtype = getattr(torch, config.model_dtype)

        self.gpt_tokenizer = GPT2TokenizerFast.from_pretrained(
            config.gpt_model
        )
        if self.gpt_tokenizer.eos_token_id is None:
            raise RuntimeError("GPT-2 tokenizer has no EOS token.")
        self.gpt_tokenizer.pad_token = self.gpt_tokenizer.eos_token

        self.gpt_model = GPT2LMHeadModel.from_pretrained(
            config.gpt_model,
            dtype=self.dtype,
            attn_implementation="eager",
        ).to(self.device)
        self.clip_processor = CLIPProcessor.from_pretrained(
    config.clip_model,
    use_fast=False,
)
        self.clip_model = CLIPModel.from_pretrained(
            config.clip_model,
            dtype=self.dtype,
            attn_implementation="eager",
        ).to(self.device)

        self.gpt_model.config.use_cache = True
        self.gpt_model.eval()
        self.clip_model.eval()
        self.gpt_model.requires_grad_(False)
        self.clip_model.requires_grad_(False)

        self._assert_models()
        self.forbidden_token_ids = self._load_forbidden_tokens()

    def _assert_models(self):
        if self.gpt_model.name_or_path != self.config.gpt_model:
            raise AssertionError("Loaded GPT-2 model ID does not match config.")
        if self.clip_model.name_or_path != self.config.clip_model:
            raise AssertionError("Loaded CLIP model ID does not match config.")
        if self.gpt_model.training or self.clip_model.training:
            raise AssertionError("Both models must be in eval mode.")
        if any(parameter.requires_grad for parameter in self.gpt_model.parameters()):
            raise AssertionError("GPT-2 is not fully frozen.")
        if any(parameter.requires_grad for parameter in self.clip_model.parameters()):
            raise AssertionError("CLIP is not fully frozen.")
        if next(self.gpt_model.parameters()).device != self.device:
            raise AssertionError("GPT-2 is on the wrong device.")
        if next(self.clip_model.parameters()).device != self.device:
            raise AssertionError("CLIP is on the wrong device.")
        if len(self.gpt_tokenizer) != 50257:
            raise AssertionError(
                "Official forbidden token IDs require the standard GPT-2 vocab "
                "of 50,257 entries."
            )

    def _validate_forbidden_array(self, path):
        path = Path(path)
        with path.open("rb") as handle:
            if handle.read(6) != b"\x93NUMPY":
                raise RuntimeError("Forbidden-token file is not a valid NPY artifact.")
        values = np.load(path, allow_pickle=False)
        if values.ndim != 1:
            raise RuntimeError("Forbidden-token array must be one-dimensional.")
        if not np.issubdtype(values.dtype, np.integer):
            raise RuntimeError("Forbidden-token array must contain integer IDs.")
        values = values.astype(np.int64, copy=False)
        if values.size == 0:
            raise RuntimeError("Forbidden-token array is empty.")
        if np.unique(values).size != values.size:
            raise RuntimeError("Forbidden-token array contains duplicate IDs.")
        if values.min() < 0 or values.max() >= len(self.gpt_tokenizer):
            raise RuntimeError(
                "Forbidden-token IDs are incompatible with the GPT-2 tokenizer."
            )
        return values

    def _load_forbidden_tokens(self):
        artifact_dir = Path(self.config.run_dir) / "artifacts"
        artifact_dir.mkdir(parents=True, exist_ok=True)
        artifact_path = artifact_dir / "forbidden_tokens.npy"
        metadata_path = artifact_dir / "forbidden_tokens_metadata.json"

        if not artifact_path.exists():
            temporary = artifact_path.with_suffix(".npy.part")
            request = urllib.request.Request(
                self.config.forbidden_tokens_url,
                headers={"User-Agent": "ZeroCap-Colab-Prototype/1.0"},
            )
            try:
                with urllib.request.urlopen(request, timeout=60) as response:
                    payload = response.read()
                with temporary.open("wb") as handle:
                    handle.write(payload)
                os.replace(temporary, artifact_path)
            except Exception as exc:
                if temporary.exists():
                    temporary.unlink()
                raise RuntimeError(
                    "Failed to download official ZeroCap forbidden_tokens.npy. "
                    "Generation is stopped; suppression will not be skipped."
                ) from exc

        try:
            values = self._validate_forbidden_array(artifact_path)
        except Exception as exc:
            raise RuntimeError(
                f"Cached forbidden-token artifact is invalid: {artifact_path}. "
                "Remove this run directory and Run all again."
            ) from exc

        sha256 = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
        metadata = {
            "source_url": self.config.forbidden_tokens_url,
            "sha256": sha256,
            "count": int(values.size),
            "tokenizer": self.config.gpt_model,
            "vocab_size": len(self.gpt_tokenizer),
        }
        PredictionStore._atomic_json(metadata_path, metadata)
        print(
            "Forbidden tokens:",
            int(values.size),
            "SHA-256:",
            sha256,
        )
        return torch.as_tensor(
            values,
            dtype=torch.long,
            device=self.device,
        )

## 8. ImageEncoder

Mặc định encode trực tiếp PIL image bằng frozen CLIP. Feature cache là optional và luôn lookup theo image_id; cache dạng image_ids + features được lập mapping ID → row, tuyệt đối không positional slice.

In [ ]:
class ImageEncoder:
    def __init__(self, models, config):
        self.models = models
        self.config = config
        self.device = models.device
        self.feature_cache = None
        self.feature_index = None
        self._load_optional_cache()

    @staticmethod
    def _feature_tensor(value):
        if torch.is_tensor(value):
            return value
        if hasattr(value, "pooler_output"):
            return value.pooler_output
        if isinstance(value, (tuple, list)) and value:
            return value[0]
        raise TypeError(f"Unsupported CLIP feature output: {type(value)}")

    def _load_optional_cache(self):
        if self.config.feature_cache_path is None:
            print("CLIP feature cache: disabled; raw images will be encoded.")
            return

        cache_path = Path(self.config.feature_cache_path)
        if not cache_path.is_file():
            raise FileNotFoundError(
                f"Configured feature cache does not exist: {cache_path}"
            )

        payload = torch.load(
            cache_path,
            map_location="cpu",
            weights_only=False,
        )
        if not isinstance(payload, dict):
            raise RuntimeError("Feature cache must be a dictionary.")

        if "image_ids" in payload and "features" in payload:
            image_ids = list(payload["image_ids"])
            features = torch.as_tensor(payload["features"])
            if len(image_ids) != features.shape[0]:
                raise RuntimeError("Cache image_ids/features length mismatch.")
            if len(image_ids) != len(set(image_ids)):
                raise RuntimeError("Feature cache contains duplicate image IDs.")
            cache_model = payload.get("clip_model")
            if cache_model is not None and cache_model != self.config.clip_model:
                raise RuntimeError(
                    f"Cache CLIP model {cache_model!r} does not match "
                    f"{self.config.clip_model!r}."
                )
            self.feature_cache = features
            self.feature_index = {
                image_id: row_index
                for row_index, image_id in enumerate(image_ids)
            }
        else:
            tensor_mapping = {
                str(image_id): torch.as_tensor(feature)
                for image_id, feature in payload.items()
                if torch.is_tensor(feature)
                or isinstance(feature, (list, tuple, np.ndarray))
            }
            if not tensor_mapping:
                raise RuntimeError(
                    "Unsupported cache schema. Expected image_ids + features "
                    "or an image_id -> feature mapping."
                )
            self.feature_cache = tensor_mapping
            self.feature_index = None

        print("CLIP feature cache loaded:", cache_path)

    def _cached_feature(self, image_id):
        if self.feature_cache is None:
            return None
        if self.feature_index is not None:
            if image_id not in self.feature_index:
                raise KeyError(f"Image ID is absent from feature cache: {image_id}")
            row_index = self.feature_index[image_id]
            feature = self.feature_cache[row_index]
        else:
            if image_id not in self.feature_cache:
                raise KeyError(f"Image ID is absent from feature cache: {image_id}")
            feature = self.feature_cache[image_id]
        return torch.as_tensor(feature)

    def encode(self, image, image_id):
        cached = self._cached_feature(image_id)
        if cached is None:
            inputs = self.models.clip_processor(
                images=image,
                return_tensors="pt",
            )
            pixel_values = inputs["pixel_values"].to(
                device=self.device,
                dtype=self.models.dtype,
            )
            with torch.no_grad():
                raw_feature = self.models.clip_model.get_image_features(
                    pixel_values=pixel_values
                )
                feature = self._feature_tensor(raw_feature)
        else:
            feature = cached.to(
                device=self.device,
                dtype=self.models.dtype,
            )
            if feature.ndim == 1:
                feature = feature.unsqueeze(0)

        if feature.ndim != 2 or feature.shape[0] != 1:
            raise AssertionError(
                f"Image feature must have shape [1, D], got {tuple(feature.shape)}"
            )
        if not torch.isfinite(feature).all():
            raise AssertionError("Image feature contains NaN/Inf.")

        feature = F.normalize(feature, dim=-1).detach()
        norm = feature.norm(dim=-1)
        if not torch.allclose(
            norm,
            torch.ones_like(norm),
            atol=1e-4,
            rtol=1e-4,
        ):
            raise AssertionError(f"Image feature norm is {norm.item()}, expected 1.")

        if self.config.debug:
            print("E_img shape:", tuple(feature.shape))
            print("E_img norm:", norm.item())
        return feature

## 9. CLIPGuidance

Mỗi Top-K candidate được tạo bằng cách nối token IDs rồi decode toàn bộ sequence một lần, tránh sai lệch byte-level BPE khi nối hai chuỗi đã decode riêng. GPT-2 BOS được kiểm tra rồi loại khỏi chuỗi nhìn thấy bởi CLIP, giống context-prefix handling của baseline. Text được encode theo chunks, L2-normalize, chuyển cosine/temperature thành target_topk, scatter đúng Top-K vào vocab và detach trước loss.

In [ ]:
class CLIPGuidance:
    def __init__(self, models, config):
        self.models = models
        self.config = config
        self.device = models.device

    @staticmethod
    def _feature_tensor(value):
        if torch.is_tensor(value):
            return value
        if hasattr(value, "pooler_output"):
            return value.pooler_output
        if isinstance(value, (tuple, list)) and value:
            return value[0]
        raise TypeError(f"Unsupported CLIP text feature output: {type(value)}")

    def _decode_for_clip(self, token_ids):
        visible_ids = [int(value) for value in token_ids]
        bos_token_id = self.models.gpt_tokenizer.bos_token_id
        leading_bos_removed = bool(
            visible_ids
            and bos_token_id is not None
            and visible_ids[0] == int(bos_token_id)
        )
        if self.config.prepend_bos_token and not leading_bos_removed:
            raise AssertionError(
                "A CLIP guidance candidate is missing the required leading BOS."
            )
        if leading_bos_removed:
            visible_ids = visible_ids[1:]
        text = self.models.gpt_tokenizer.decode(
            visible_ids,
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )
        return text, leading_bos_removed

    def encode_texts(self, texts):
        all_features = []
        chunk_size = self.config.clip_candidate_chunk_size
        for start in range(0, len(texts), chunk_size):
            chunk = list(texts[start: start + chunk_size])
            inputs = self.models.clip_processor(
                text=chunk,
                padding=True,
                truncation=True,
                max_length=self.config.clip_max_text_tokens,
                return_tensors="pt",
            )
            text_inputs = {
                key: value.to(self.device)
                for key, value in inputs.items()
                if key in {"input_ids", "attention_mask"}
            }
            with torch.no_grad():
                output = self.models.clip_model.get_text_features(
                    **text_inputs
                )
                features = self._feature_tensor(output)
                features = F.normalize(features, dim=-1)
            if not torch.isfinite(features).all():
                raise AssertionError("CLIP text features contain NaN/Inf.")
            all_features.append(features.detach())

        encoded = torch.cat(all_features, dim=0)
        if encoded.shape[0] != len(texts):
            raise AssertionError("CLIP text feature count mismatch.")
        return encoded

    def build_target(
        self,
        image_feature,
        current_ids,
        top_indices,
        vocabulary_size,
        target_dtype,
    ):
        tokenizer = self.models.gpt_tokenizer
        current_token_ids = [
            int(value)
            for value in current_ids[0].detach().cpu().tolist()
        ]
        current_text, current_bos_removed = self._decode_for_clip(
            current_token_ids
        )

        candidate_texts = []
        decoded_tokens = []
        candidate_text_prefix_matches = []
        for token_id in top_indices[0].detach().cpu().tolist():
            token_id = int(token_id)
            token_text = tokenizer.decode(
                [token_id],
                skip_special_tokens=False,
                clean_up_tokenization_spaces=False,
            )
            candidate_token_ids = [*current_token_ids, token_id]
            candidate, candidate_bos_removed = self._decode_for_clip(
                candidate_token_ids
            )
            if not candidate_bos_removed:
                raise AssertionError("Leading BOS was not removed before CLIP.")
            if candidate_token_ids[:-1] != current_token_ids:
                raise AssertionError(
                    "Candidate token sequence lost the full current prefix."
                )
            candidate_texts.append(candidate)
            decoded_tokens.append(token_text)
            candidate_text_prefix_matches.append(
                candidate.startswith(current_text)
            )

        text_features = self.encode_texts(candidate_texts)
        if text_features.shape[-1] != image_feature.shape[-1]:
            raise AssertionError(
                "CLIP image/text projection dimensions do not match."
            )
        similarities = (
            text_features @ image_feature.detach().transpose(0, 1)
        ).squeeze(1)
        if not torch.isfinite(similarities).all():
            raise AssertionError("CLIP candidate similarities contain NaN/Inf.")

        target_topk = torch.softmax(
            similarities / self.config.clip_temperature,
            dim=0,
        ).detach()
        target_vocab = torch.zeros(
            (1, vocabulary_size),
            device=self.device,
            dtype=target_dtype,
        )
        target_vocab.scatter_(
            dim=1,
            index=top_indices,
            src=target_topk.unsqueeze(0).to(target_dtype),
        )
        target_vocab = target_vocab.detach()

        if not torch.allclose(
            target_vocab.sum(),
            torch.ones((), device=self.device, dtype=target_dtype),
            atol=1e-4,
            rtol=1e-4,
        ):
            raise AssertionError("target_vocab does not sum to one.")

        outside_topk = target_vocab.clone()
        outside_topk.scatter_(
            1,
            top_indices,
            torch.zeros_like(top_indices, dtype=target_dtype),
        )
        if torch.count_nonzero(outside_topk).item() != 0:
            raise AssertionError("CLIP target is non-zero outside Top-K.")

        diagnostics = {
            "candidate_construction": "full_sequence_decode",
            "leading_bos_removed_before_clip": current_bos_removed,
            "current_text": current_text,
            "decoded_top_tokens": decoded_tokens,
            "candidate_texts": candidate_texts,
            "candidate_text_prefix_matches": (
                candidate_text_prefix_matches
            ),
            "similarities": [
                float(value)
                for value in similarities.detach().cpu().tolist()
            ],
            "target_sum": float(target_vocab.sum().item()),
        }
        return target_vocab, diagnostics

    def rerank(
        self,
        image_feature,
        candidate_texts,
        normalized_beam_scores,
    ):
        if len(candidate_texts) != len(normalized_beam_scores):
            raise AssertionError("Final rerank candidate/score count mismatch.")
        if not candidate_texts:
            raise AssertionError("Final rerank requires at least one candidate.")
        if not all(str(text).strip() for text in candidate_texts):
            raise AssertionError("Final rerank received an empty caption.")
        text_features = self.encode_texts(candidate_texts)
        similarities = (
            text_features @ image_feature.detach().transpose(0, 1)
        ).squeeze(1)
        if not torch.isfinite(similarities).all():
            raise AssertionError("Final CLIP reranking produced NaN/Inf.")

        beam_scores = torch.tensor(
            normalized_beam_scores,
            device=similarities.device,
            dtype=similarities.dtype,
        )
        if not torch.isfinite(beam_scores).all():
            raise AssertionError("Final beam scores contain NaN/Inf.")

        clip_component = (
            self.config.final_rerank_clip_weight * similarities
        )
        if self.config.final_rerank_mode == "clip_only":
            rerank_scores = clip_component
        elif self.config.final_rerank_mode == "clip_beam":
            rerank_scores = (
                clip_component
                + self.config.final_rerank_beam_weight * beam_scores
            )
        else:
            raise AssertionError(
                f"Unsupported final rerank mode: "
                f"{self.config.final_rerank_mode!r}."
            )
        if not torch.isfinite(rerank_scores).all():
            raise AssertionError("Final rerank scores contain NaN/Inf.")

        best_index = int(torch.argmax(rerank_scores).item())
        clip_only_index = int(torch.argmax(similarities).item())
        beam_only_index = int(torch.argmax(beam_scores).item())
        return (
            best_index,
            [float(value) for value in similarities.detach().cpu().tolist()],
            [float(value) for value in beam_scores.detach().cpu().tolist()],
            [float(value) for value in rerank_scores.detach().cpu().tolist()],
            clip_only_index,
            beam_only_index,
        )

## 10. ContextOptimizer — legacy past-key/value delta optimization

Base past được tạo bằng no_grad (không dùng inference_mode). Mỗi layer có delta_key/delta_value từ zeros_like. Guided GPT forward giữ autograd chỉ tới delta; sau backward gradient được normalize theo từng tensor, update thủ công, detach và tạo leaf mới. Notebook đánh giá N trạng thái trước update cùng trạng thái sau update cuối; best observed loss được lưu để chẩn đoán nhưng generation dùng detached delta sau update cuối đúng vòng lặp ZeroCap, vì Top-K/CLIP target thay đổi giữa iteration nên các loss không phải cùng một objective cố định để checkpoint-select.

In [ ]:
class ContextOptimizer:
    def __init__(self, models, guidance, config):
        self.models = models
        self.guidance = guidance
        self.config = config
        self.gpt = models.gpt_model

    def _legacy_cache(self, cache):
        if hasattr(cache, "to_legacy_cache"):
            cache = cache.to_legacy_cache()
        if not isinstance(cache, (tuple, list)):
            raise RuntimeError(
                "Pinned Transformers did not return a legacy-compatible cache."
            )
        legacy = []
        for layer_index, layer in enumerate(cache):
            if not isinstance(layer, (tuple, list)) or len(layer) != 2:
                raise AssertionError(
                    f"Invalid cache structure at layer {layer_index}."
                )
            key, value = layer
            if not torch.is_tensor(key) or not torch.is_tensor(value):
                raise AssertionError("Cache key/value must be tensors.")
            if key.ndim != 4 or value.ndim != 4:
                raise AssertionError("GPT-2 cache tensors must be four-dimensional.")
            if key.shape != value.shape:
                raise AssertionError("GPT-2 key/value shapes do not match.")
            legacy.append((key.detach(), value.detach()))

        expected_layers = int(self.gpt.config.n_layer)
        if len(legacy) != expected_layers:
            raise AssertionError(
                f"Cache has {len(legacy)} layers, expected {expected_layers}."
            )
        return tuple(legacy)

    @staticmethod
    def _cache_for_forward(legacy_cache):
        from transformers import DynamicCache

        return DynamicCache.from_legacy_cache(
            tuple(legacy_cache)
        )

    def _new_delta(self, past_original, previous_delta=None):
        use_previous = (
            previous_delta is not None
            and not self.config.reset_context_delta
        )
        if use_previous and len(previous_delta) != len(past_original):
            raise AssertionError("Previous delta layer count mismatch.")

        delta = []
        for layer_index, (key, value) in enumerate(past_original):
            delta_key = torch.zeros_like(key)
            delta_value = torch.zeros_like(value)

            if use_previous:
                previous_key, previous_value = previous_delta[layer_index]
                for destination, source in (
                    (delta_key, previous_key),
                    (delta_value, previous_value),
                ):
                    if (
                        destination.device != source.device
                        or destination.dtype != source.dtype
                        or destination.shape[:-2] != source.shape[:-2]
                        or destination.shape[-1] != source.shape[-1]
                    ):
                        raise AssertionError(
                            "Cannot carry context delta across incompatible cache."
                        )
                    copied_length = min(
                        destination.shape[-2],
                        source.shape[-2],
                    )
                    destination[..., :copied_length, :].copy_(
                        source.detach()[..., :copied_length, :]
                    )

            delta_key.requires_grad_(True)
            delta_value.requires_grad_(True)
            delta.append((delta_key, delta_value))

        result = tuple(delta)
        self._assert_delta(past_original, result, require_grad=True)
        return result

    @staticmethod
    def _add_delta(past_original, delta):
        return tuple(
            (
                original_key + delta_key,
                original_value + delta_value,
            )
            for (
                (original_key, original_value),
                (delta_key, delta_value),
            ) in zip(past_original, delta)
        )

    @staticmethod
    def _assert_delta(past_original, delta, require_grad):
        if len(past_original) != len(delta):
            raise AssertionError("Context delta structure mismatch.")
        for layer_index, (
            (original_key, original_value),
            (delta_key, delta_value),
        ) in enumerate(zip(past_original, delta)):
            for original, candidate, name in (
                (original_key, delta_key, "key"),
                (original_value, delta_value, "value"),
            ):
                if candidate.shape != original.shape:
                    raise AssertionError(
                        f"Delta {name} shape mismatch at layer {layer_index}."
                    )
                if candidate.device != original.device:
                    raise AssertionError("Delta device mismatch.")
                if candidate.dtype != original.dtype:
                    raise AssertionError("Delta dtype mismatch.")
                if require_grad and not candidate.requires_grad:
                    raise AssertionError("Context delta must require gradients.")

    def _original_distribution_and_past(self, current_ids):
        if current_ids.ndim != 2 or current_ids.shape[0] != 1:
            raise AssertionError("current_ids must have shape [1, sequence].")
        if current_ids.shape[1] < 2:
            raise AssertionError(
                "Prompt/current sequence must contain at least two GPT-2 tokens."
            )

        with torch.no_grad():
            original_output = self.gpt(
                input_ids=current_ids,
                use_cache=False,
                return_dict=True,
            )
            p_original = torch.softmax(
                original_output.logits[:, -1, :],
                dim=-1,
            ).detach()

            prefix_ids = current_ids[:, :-1]
            prefix_output = self.gpt(
                input_ids=prefix_ids,
                use_cache=True,
                return_dict=True,
            )
            past_original = self._legacy_cache(
                prefix_output.past_key_values
            )

        assert_probability_distribution(p_original, "P_original")
        if p_original.requires_grad or p_original.grad_fn is not None:
            raise AssertionError("P_original must be fully detached.")
        return p_original, past_original, current_ids[:, -1:]

    def _assert_zero_delta_cache_equivalence(
        self,
        p_original,
        past_original,
        last_token,
    ):
        with torch.no_grad():
            cached_output = self.gpt(
                input_ids=last_token,
                past_key_values=self._cache_for_forward(past_original),
                use_cache=False,
                return_dict=True,
            )
            p_cached = torch.softmax(
                cached_output.logits[:, -1, :],
                dim=-1,
            ).detach()

        assert_probability_distribution(p_cached, "P_cached_zero_delta")
        absolute_error = (p_cached - p_original).abs()
        max_abs_error = float(absolute_error.max().item())
        mean_abs_error = float(absolute_error.mean().item())
        original_top_token_id = int(p_original.argmax(dim=-1).item())
        cached_top_token_id = int(p_cached.argmax(dim=-1).item())
        same_top_token = original_top_token_id == cached_top_token_id

        if not torch.allclose(
            p_cached,
            p_original,
            atol=1e-4,
            rtol=1e-4,
        ):
            raise AssertionError(
                "Zero-delta cached GPT-2 distribution does not match "
                "the full-sequence P_original; stop before interpreting "
                "CLIP guidance results. "
                f"max_abs_error={max_abs_error:.6g}, "
                f"mean_abs_error={mean_abs_error:.6g}."
            )

        return {
            "max_abs_error": max_abs_error,
            "mean_abs_error": mean_abs_error,
            "original_top_token_id": original_top_token_id,
            "cached_top_token_id": cached_top_token_id,
            "same_top_token": same_top_token,
            "p_original_sum": float(p_original.sum().item()),
            "p_cached_sum": float(p_cached.sum().item()),
        }

    def _evaluate_delta(
        self,
        current_ids,
        image_feature,
        p_original,
        past_original,
        last_token,
        delta,
    ):
        past_guided = self._add_delta(past_original, delta)
        guided_output = self.gpt(
            input_ids=last_token,
            past_key_values=self._cache_for_forward(past_guided),
            use_cache=False,
            return_dict=True,
        )
        p_guided = torch.softmax(
            guided_output.logits[:, -1, :],
            dim=-1,
        )
        assert_probability_distribution(p_guided, "P_guided")

        top_count = min(self.config.top_k, p_guided.shape[-1])
        top_probabilities, top_indices = torch.topk(
            p_guided,
            k=top_count,
            dim=-1,
        )
        target_vocab, clip_diagnostics = self.guidance.build_target(
            image_feature=image_feature,
            current_ids=current_ids,
            top_indices=top_indices,
            vocabulary_size=p_guided.shape[-1],
            target_dtype=p_guided.dtype,
        )
        if target_vocab.requires_grad or target_vocab.grad_fn is not None:
            raise AssertionError("CLIP target must be detached.")

        log_guided = torch.log(
            p_guided.clamp_min(self.config.probability_eps)
        )
        log_original = torch.log(
            p_original.clamp_min(self.config.probability_eps)
        )
        loss_clip = -(target_vocab * log_guided).sum()
        loss_fluency = (
            p_guided * (log_guided - log_original)
        ).sum()
        loss_total = (
            self.config.clip_loss_scale * loss_clip
            + self.config.fluency_weight * loss_fluency
        )

        for label, loss in (
            ("CLIP loss", loss_clip),
            ("fluency loss", loss_fluency),
            ("total loss", loss_total),
        ):
            if not torch.isfinite(loss):
                raise AssertionError(f"{label} is NaN/Inf.")

        return {
            "p_guided": p_guided,
            "top_probabilities": top_probabilities,
            "top_indices": top_indices,
            "target_vocab": target_vocab,
            "clip_diagnostics": clip_diagnostics,
            "loss_clip": loss_clip,
            "loss_fluency": loss_fluency,
            "loss_total": loss_total,
        }

    def optimize(
        self,
        current_ids,
        image_feature,
        previous_delta=None,
        trace=None,
        trace_context=None,
    ):
        p_original, past_original, last_token = (
            self._original_distribution_and_past(current_ids)
        )
        delta = self._new_delta(
            past_original,
            previous_delta=previous_delta,
        )
        diagnostics = []
        trace_context = dict(trace_context or {})

        cache_equivalence = None
        if (
            trace is not None
            and trace_context.get("step") == 1
            and trace_context.get("beam_index") == 0
        ):
            cache_equivalence = (
                self._assert_zero_delta_cache_equivalence(
                    p_original=p_original,
                    past_original=past_original,
                    last_token=last_token,
                )
            )
            trace.record(
                "zero_delta_cache_equivalence",
                context=trace_context,
                **cache_equivalence,
            )

        if trace is not None:
            first_key, first_value = past_original[0]
            trace.record(
                "context_initialized",
                context=trace_context,
                current_token_ids=[
                    int(value)
                    for value in current_ids.detach().cpu().reshape(-1).tolist()
                ],
                current_text=self.models.gpt_tokenizer.decode(
                    current_ids.detach().cpu().reshape(-1).tolist(),
                    skip_special_tokens=False,
                    clean_up_tokenization_spaces=False,
                ),
                p_original=probability_diagnostics(
                    p_original,
                    self.models.gpt_tokenizer,
                    self.config.diagnostic_top_n,
                ),
                cache={
                    "layers": len(past_original),
                    "first_key_shape": list(first_key.shape),
                    "first_value_shape": list(first_value.shape),
                    "dtype": str(first_key.dtype),
                    "device": str(first_key.device),
                },
                initial_delta=context_delta_diagnostics(delta),
            )

        if self.config.debug:
            first_key, first_value = past_original[0]
            print("past_key_values layers:", len(past_original))
            print("past[0] key/value:", tuple(first_key.shape), tuple(first_value.shape))
            print("delta[0] key/value:", tuple(delta[0][0].shape), tuple(delta[0][1].shape))
            print("P_original sum:", float(p_original.sum().item()))
            if cache_equivalence is not None:
                print(
                    "zero-delta cache max/mean abs error:",
                    cache_equivalence["max_abs_error"],
                    cache_equivalence["mean_abs_error"],
                )

        best_loss = math.inf
        best_iteration = None

        for iteration in range(self.config.inner_iterations):
            delta_before = context_delta_diagnostics(delta)
            evaluation = self._evaluate_delta(
                current_ids=current_ids,
                image_feature=image_feature,
                p_original=p_original,
                past_original=past_original,
                last_token=last_token,
                delta=delta,
            )
            p_guided = evaluation["p_guided"]
            top_probabilities = evaluation["top_probabilities"]
            top_indices = evaluation["top_indices"]
            target_vocab = evaluation["target_vocab"]
            clip_diagnostics = evaluation["clip_diagnostics"]
            loss_clip = evaluation["loss_clip"]
            loss_fluency = evaluation["loss_fluency"]
            loss_total = evaluation["loss_total"]

            loss_value = float(loss_total.detach().item())
            is_best_so_far = loss_value < best_loss
            if is_best_so_far:
                best_loss = loss_value
                best_iteration = iteration

            loss_total.backward()

            updated_layers = []
            gradient_norms = []
            for delta_key, delta_value in delta:
                updated_pair = []
                for delta_tensor in (delta_key, delta_value):
                    gradient = delta_tensor.grad
                    if gradient is None:
                        raise AssertionError("Context delta has no gradient.")
                    if not torch.isfinite(gradient).all():
                        raise AssertionError("Context delta gradient is NaN/Inf.")
                    gradient_norm = gradient.norm()
                    normalized_gradient = gradient / (
                        (gradient_norm + self.config.gradient_eps)
                        ** self.config.grad_norm_factor
                    )
                    updated = (
                        delta_tensor
                        - self.config.step_size * normalized_gradient
                    ).detach()
                    updated.requires_grad_(True)
                    updated_pair.append(updated)
                    gradient_norms.append(float(gradient_norm.item()))
                updated_layers.append(tuple(updated_pair))
            delta = tuple(updated_layers)
            self._assert_delta(past_original, delta, require_grad=True)

            assert_model_parameter_grads_clear(
                self.models.gpt_model,
                "GPT-2",
            )
            assert_model_parameter_grads_clear(
                self.models.clip_model,
                "CLIP",
            )

            iteration_diagnostics = {
                "iteration": iteration,
                "evaluation_type": "pre_update",
                "is_best_so_far": is_best_so_far,
                "loss_clip": float(loss_clip.detach().item()),
                "loss_fluency": float(loss_fluency.detach().item()),
                "loss_total": loss_value,
                "p_guided_sum": float(p_guided.detach().sum().item()),
                "gradient_norm_min": min(gradient_norms),
                "gradient_norm_max": max(gradient_norms),
                "gradient_norm_mean": float(np.mean(gradient_norms)),
                "top_token_ids": [
                    int(value)
                    for value in top_indices[0].detach().cpu().tolist()
                ],
                "top_guided_probabilities": [
                    float(value)
                    for value in top_probabilities[0].detach().cpu().tolist()
                ],
                "target_topk_probabilities": [
                    float(value)
                    for value in target_vocab[0, top_indices[0]].detach().cpu().tolist()
                ],
                "delta_before": delta_before,
                "delta_after": context_delta_diagnostics(delta),
                **clip_diagnostics,
            }
            diagnostics.append(iteration_diagnostics)

            if trace is not None:
                trace.record(
                    "inner_optimization_iteration",
                    context=trace_context,
                    diagnostics=compact_iteration_diagnostics(
                        iteration_diagnostics,
                        self.config.diagnostic_top_n,
                        self.config.diagnostic_save_all_top_k,
                    ),
                    p_guided=probability_diagnostics(
                        p_guided,
                        self.models.gpt_tokenizer,
                        self.config.diagnostic_top_n,
                    ),
                )

            if self.config.debug:
                print(
                    f"inner={iteration} "
                    f"clip={iteration_diagnostics['loss_clip']:.6f} "
                    f"fluency={iteration_diagnostics['loss_fluency']:.6f} "
                    f"total={iteration_diagnostics['loss_total']:.6f}"
                )
                print(
                    "Top-K decoded candidates:",
                    iteration_diagnostics["candidate_texts"],
                )
                print(
                    "CLIP similarities:",
                    iteration_diagnostics["similarities"],
                )
                print(
                    "P_guided sum:",
                    iteration_diagnostics["p_guided_sum"],
                )

        del (
            evaluation,
            p_guided,
            top_probabilities,
            top_indices,
            target_vocab,
            loss_clip,
            loss_fluency,
            loss_total,
        )

        post_update_delta = context_delta_diagnostics(delta)
        post_update_evaluation = self._evaluate_delta(
            current_ids=current_ids,
            image_feature=image_feature,
            p_original=p_original,
            past_original=past_original,
            last_token=last_token,
            delta=delta,
        )
        post_p_guided = post_update_evaluation["p_guided"]
        post_top_probabilities = post_update_evaluation[
            "top_probabilities"
        ]
        post_top_indices = post_update_evaluation["top_indices"]
        post_target_vocab = post_update_evaluation["target_vocab"]
        post_clip_diagnostics = post_update_evaluation[
            "clip_diagnostics"
        ]
        post_loss_clip = post_update_evaluation["loss_clip"]
        post_loss_fluency = post_update_evaluation["loss_fluency"]
        post_loss_total = post_update_evaluation["loss_total"]
        post_loss_value = float(post_loss_total.detach().item())
        post_is_best = post_loss_value < best_loss
        if post_is_best:
            best_loss = post_loss_value
            best_iteration = self.config.inner_iterations

        post_update_diagnostics = {
            "iteration": self.config.inner_iterations,
            "evaluation_type": "post_final_update",
            "is_best_so_far": post_is_best,
            "loss_clip": float(post_loss_clip.detach().item()),
            "loss_fluency": float(post_loss_fluency.detach().item()),
            "loss_total": post_loss_value,
            "p_guided_sum": float(post_p_guided.detach().sum().item()),
            "gradient_norm_min": None,
            "gradient_norm_max": None,
            "gradient_norm_mean": None,
            "top_token_ids": [
                int(value)
                for value in post_top_indices[0].detach().cpu().tolist()
            ],
            "top_guided_probabilities": [
                float(value)
                for value in post_top_probabilities[0].detach().cpu().tolist()
            ],
            "target_topk_probabilities": [
                float(value)
                for value in post_target_vocab[
                    0, post_top_indices[0]
                ].detach().cpu().tolist()
            ],
            "delta_before": post_update_delta,
            "delta_after": post_update_delta,
            **post_clip_diagnostics,
        }
        diagnostics.append(post_update_diagnostics)

        if trace is not None:
            trace.record(
                "inner_optimization_post_update_evaluation",
                context=trace_context,
                diagnostics=compact_iteration_diagnostics(
                    post_update_diagnostics,
                    self.config.diagnostic_top_n,
                    self.config.diagnostic_save_all_top_k,
                ),
                p_guided=probability_diagnostics(
                    post_p_guided,
                    self.models.gpt_tokenizer,
                    self.config.diagnostic_top_n,
                ),
            )

        if self.config.debug:
            print(
                f"post-update clip={post_update_diagnostics['loss_clip']:.6f} "
                f"fluency={post_update_diagnostics['loss_fluency']:.6f} "
                f"total={post_update_diagnostics['loss_total']:.6f}"
            )
            print("Diagnostic best delta evaluation:", best_iteration)
            print(
                "Generation uses post-final-update delta evaluation:",
                self.config.inner_iterations,
            )

        if best_iteration is None:
            raise AssertionError("No valid context delta evaluation was recorded.")
        detached_delta = tuple(
            (
                delta_key.detach(),
                delta_value.detach(),
            )
            for delta_key, delta_value in delta
        )
        self._assert_delta(
            past_original,
            detached_delta,
            require_grad=False,
        )
        for delta_key, delta_value in detached_delta:
            for delta_tensor in (delta_key, delta_value):
                if delta_tensor.requires_grad or delta_tensor.grad_fn is not None:
                    raise AssertionError("Final context delta retained a graph.")

        del (
            post_update_evaluation,
            post_p_guided,
            post_top_probabilities,
            post_top_indices,
            post_target_vocab,
            post_loss_clip,
            post_loss_fluency,
            post_loss_total,
        )
        past_final = self._add_delta(
            past_original,
            detached_delta,
        )
        with torch.no_grad():
            final_output = self.gpt(
                input_ids=last_token,
                past_key_values=self._cache_for_forward(past_final),
                use_cache=False,
                return_dict=True,
            )
            p_guided_final = torch.softmax(
                final_output.logits[:, -1, :],
                dim=-1,
            ).detach()
        assert_probability_distribution(
            p_guided_final,
            "P_guided_final",
        )

        if trace is not None:
            trace.record(
                "context_optimized",
                context=trace_context,
                best_delta_evaluation=int(best_iteration),
                best_loss_total=float(best_loss),
                selected_delta_evaluation=(
                    self.config.inner_iterations
                ),
                selected_loss_total=float(post_loss_value),
                selected_minus_best_loss=float(
                    post_loss_value - best_loss
                ),
                evaluated_delta_states=(
                    self.config.inner_iterations + 1
                ),
                p_guided_final=probability_diagnostics(
                    p_guided_final,
                    self.models.gpt_tokenizer,
                    self.config.diagnostic_top_n,
                ),
                final_delta=context_delta_diagnostics(detached_delta),
            )

        if self.config.debug:
            print("P_guided_final sum:", float(p_guided_final.sum().item()))
            print(
                "Diagnostic best delta evaluation/loss:",
                best_iteration,
                best_loss,
            )
            print(
                "Selected final-update evaluation/loss:",
                self.config.inner_iterations,
                post_loss_value,
            )

        return ContextStepResult(
            p_original=p_original,
            p_guided_final=p_guided_final,
            context_delta=detached_delta,
            diagnostics=diagnostics,
        )

## 11. ZeroCapDecoder — fusion and special-token policy

Decoder áp dụng forbidden suppression, repetition penalty, hard-mask period/EOS trước min length và period boost lên guided distribution trước. Sau đó mới geometric-fuse với original GPT-2 distribution trong log-space; hard early-stop mask được giữ lại qua fusion để bảo đảm min length, với EOS fallback rõ ràng.

In [ ]:
class ZeroCapDecoder:
    VALID_STOP_REASONS = {
        "period",
        "eos",
        "eos_fallback",
        "max_tokens",
    }

    def __init__(self, models, config):
        self.models = models
        self.config = config
        self.tokenizer = models.gpt_tokenizer
        self.eos_token_id = int(self.tokenizer.eos_token_id)

        stop_ids = set(
            self.tokenizer.encode(
                config.stop_token,
                add_special_tokens=False,
            )
        )
        stop_ids.update(
            self.tokenizer.encode(
                " " + config.stop_token,
                add_special_tokens=False,
            )
        )
        if not stop_ids:
            raise RuntimeError("stop_token did not map to any GPT-2 token.")
        self.stop_token_ids = {
            int(token_id) for token_id in stop_ids
        }

        forbidden = set(
            int(value)
            for value in models.forbidden_token_ids.detach().cpu().tolist()
        )
        forbidden.discard(self.eos_token_id)
        forbidden.difference_update(self.stop_token_ids)
        self.forbidden_token_ids = sorted(forbidden)

    def fuse(
        self,
        p_guided_constrained,
        p_original,
        hard_suppressed_token_ids=(),
    ):
        log_guided = torch.log(
            p_guided_constrained.clamp_min(self.config.probability_eps)
        )
        log_original = torch.log(
            p_original.clamp_min(self.config.probability_eps)
        )
        log_p_final = (
            self.config.fusion_factor * log_guided
            + (1.0 - self.config.fusion_factor) * log_original
        )
        if hard_suppressed_token_ids:
            hard_suppressed = torch.tensor(
                sorted(set(int(value) for value in hard_suppressed_token_ids)),
                dtype=torch.long,
                device=log_p_final.device,
            )
            log_p_final[:, hard_suppressed] = -torch.inf
        if not torch.isfinite(log_p_final).any():
            raise RuntimeError("Fusion suppressed every vocabulary token.")
        p_final = torch.softmax(log_p_final, dim=-1)
        assert_probability_distribution(p_final, "P_fused")
        return p_final

    def apply_guided_constraints(
        self,
        p_guided_final,
        generated_token_ids,
    ):
        before = probability_diagnostics(
            p_guided_final,
            self.tokenizer,
            self.config.diagnostic_top_n,
        )
        log_probabilities = torch.log(
            p_guided_final.clamp_min(self.config.probability_eps)
        ).clone()
        generated_count = len(generated_token_ids)
        repeated_token_ids = []
        early_stop_ids = []
        stop_boost = 0.0

        if self.forbidden_token_ids:
            forbidden_tensor = torch.tensor(
                self.forbidden_token_ids,
                dtype=torch.long,
                device=log_probabilities.device,
            )
            log_probabilities[:, forbidden_tensor] -= math.log(
                self.config.forbidden_factor
            )

        if (
            self.config.repetition_penalty > 1.0
            and generated_token_ids
        ):
            repeated = torch.tensor(
                sorted(set(generated_token_ids)),
                dtype=torch.long,
                device=log_probabilities.device,
            )
            log_probabilities[:, repeated] -= math.log(
                self.config.repetition_penalty
            )
            repeated_token_ids = [int(value) for value in repeated.tolist()]

        if generated_count < self.config.min_new_tokens:
            early_stop_ids = sorted(
                self.stop_token_ids | {self.eos_token_id}
            )
            log_probabilities[:, early_stop_ids] = -torch.inf
        else:
            boost_power = (
                generated_count
                - self.config.min_new_tokens
                + 1
            )
            stop_boost = boost_power * math.log(self.config.end_factor)
            for token_id in self.stop_token_ids:
                log_probabilities[:, token_id] += stop_boost

        if not torch.isfinite(log_probabilities).any():
            fallback = torch.zeros_like(p_guided_final)
            fallback[:, self.eos_token_id] = 1.0
            assert_probability_distribution(fallback, "P_final_eos_fallback")
            diagnostics = {
                "order": "guided_constraints_before_fusion",
                "used_fallback": True,
                "fallback_reason": "no_finite_constrained_logits",
                "generated_count": generated_count,
                "forbidden_token_count": len(self.forbidden_token_ids),
                "repeated_token_ids": repeated_token_ids,
                "early_stop_ids": early_stop_ids,
                "hard_suppressed_token_ids": early_stop_ids,
                "stop_boost_log": float(stop_boost),
                "before_guided_constraints": before,
                "after_guided_constraints": probability_diagnostics(
                    fallback, self.tokenizer, self.config.diagnostic_top_n
                ),
            }
            return fallback, True, diagnostics

        constrained = torch.softmax(log_probabilities, dim=-1)
        if not torch.isfinite(constrained).all() or constrained.sum() <= 0:
            fallback = torch.zeros_like(p_guided_final)
            fallback[:, self.eos_token_id] = 1.0
            assert_probability_distribution(fallback, "P_final_eos_fallback")
            diagnostics = {
                "order": "guided_constraints_before_fusion",
                "used_fallback": True,
                "fallback_reason": "invalid_constrained_distribution",
                "generated_count": generated_count,
                "forbidden_token_count": len(self.forbidden_token_ids),
                "repeated_token_ids": repeated_token_ids,
                "early_stop_ids": early_stop_ids,
                "hard_suppressed_token_ids": early_stop_ids,
                "stop_boost_log": float(stop_boost),
                "before_guided_constraints": before,
                "after_guided_constraints": probability_diagnostics(
                    fallback, self.tokenizer, self.config.diagnostic_top_n
                ),
            }
            return fallback, True, diagnostics

        assert_probability_distribution(
            constrained,
            "P_guided_constrained",
        )
        diagnostics = {
            "order": "guided_constraints_before_fusion",
            "used_fallback": False,
            "fallback_reason": "",
            "generated_count": generated_count,
            "forbidden_token_count": len(self.forbidden_token_ids),
            "forbidden_factor": float(self.config.forbidden_factor),
            "repetition_penalty": float(self.config.repetition_penalty),
            "repeated_token_ids": repeated_token_ids,
            "early_stop_ids": [int(value) for value in early_stop_ids],
            "hard_suppressed_token_ids": [
                int(value) for value in early_stop_ids
            ],
            "stop_boost_log": float(stop_boost),
            "before_guided_constraints": before,
            "after_guided_constraints": probability_diagnostics(
                constrained, self.tokenizer, self.config.diagnostic_top_n
            ),
        }
        return constrained, False, diagnostics

    def stop_reason_for_token(self, token_id, used_fallback):
        token_id = int(token_id)
        if used_fallback and token_id == self.eos_token_id:
            return "eos_fallback"
        if token_id in self.stop_token_ids:
            return "period"
        if token_id == self.eos_token_id:
            return "eos"
        return ""

    def decode_full(self, token_ids):
        flat_token_ids = (
            token_ids.detach()
            .cpu()
            .reshape(-1)
            .tolist()
        )

        return self.tokenizer.decode(
            flat_token_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

    def caption_from_full(self, full_text, prompt_text, generated_token_ids):
        if full_text.startswith(prompt_text):
            caption = full_text[len(prompt_text):]
        else:
            caption = self.tokenizer.decode(
                list(generated_token_ids),
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )
        return postprocess_caption(caption, self.config.prompt)

## 12. ZeroCapGenerator — true beam search and auditable final reranking

Mỗi active beam tối ưu context riêng. GPT-2 context bắt đầu bằng BOS trước prompt; Child beams nhận deep detached clone của delta và stopped beams được giữ nguyên. Pruning dùng accumulated log-prob chia length^beam_length_penalty. Baseline mặc định `clip_only` và final CLIP rerank chỉ encode phần caption đã bỏ prompt, theo code ZeroCap chính thức. `clip_beam` và `prompt_plus_caption` chỉ được giữ làm ablation tùy chọn. Trace lưu full text, caption-only rerank text, CLIP-only, beam-only và lựa chọn cuối; không dùng ground truth.

In [ ]:
class ZeroCapGenerator:
    def __init__(
        self,
        models,
        guidance,
        context_optimizer,
        decoder,
        config,
    ):
        self.models = models
        self.guidance = guidance
        self.context_optimizer = context_optimizer
        self.decoder = decoder
        self.config = config

    def _prompt_ids(self):
        prompt_only_ids = self.models.gpt_tokenizer(
            self.config.prompt,
            return_tensors="pt",
            add_special_tokens=False,
        )["input_ids"].to(self.models.device)
        if prompt_only_ids.shape[1] < 2:
            raise AssertionError(
                "Prompt must tokenize to at least two GPT-2 tokens."
            )
        if not self.config.prepend_bos_token:
            return prompt_only_ids
        bos_token_id = self.models.gpt_tokenizer.bos_token_id
        if bos_token_id is None:
            raise RuntimeError("GPT-2 tokenizer has no BOS token.")
        bos_ids = torch.full(
            (1, 1),
            int(bos_token_id),
            dtype=prompt_only_ids.dtype,
            device=prompt_only_ids.device,
        )
        encoded = torch.cat([bos_ids, prompt_only_ids], dim=1)
        if int(encoded[0, 0].item()) != int(bos_token_id):
            raise AssertionError("BOS token was not prepended to the prompt.")
        return encoded

    def generate(self, image_feature, trace=None):
        prompt_ids = self._prompt_ids()
        prompt_text = self.decoder.decode_full(prompt_ids)
        beams = [BeamState(token_ids=prompt_ids)]
        loop_iterations = 0

        if trace is not None:
            trace.record(
                "generation_started",
                prompt=self.config.prompt,
                prompt_text=prompt_text,
                prompt_token_ids=[
                    int(value)
                    for value in prompt_ids.detach().cpu().reshape(-1).tolist()
                ],
                profile={
                    "prepend_bos_token": self.config.prepend_bos_token,
                    "bos_token_id": (
                        int(self.models.gpt_tokenizer.bos_token_id)
                        if self.models.gpt_tokenizer.bos_token_id is not None
                        else None
                    ),
                    "top_k": self.config.top_k,
                    "inner_iterations": self.config.inner_iterations,
                    "beam_size": self.config.beam_size,
                    "max_new_tokens": self.config.max_new_tokens,
                    "fusion_factor": self.config.fusion_factor,
                    "fluency_weight": self.config.fluency_weight,
                    "final_rerank_mode": (
                        self.config.final_rerank_mode
                    ),
                    "final_rerank_text_mode": (
                        self.config.final_rerank_text_mode
                    ),
                    "final_rerank_clip_weight": (
                        self.config.final_rerank_clip_weight
                    ),
                    "final_rerank_beam_weight": (
                        self.config.final_rerank_beam_weight
                    ),
                },
            )

        for step_index in range(self.config.max_new_tokens):
            loop_iterations += 1
            candidates = []

            for beam_index, beam in enumerate(beams):
                if beam.stopped:
                    candidates.append(beam)
                    continue

                step_result = self.context_optimizer.optimize(
                    current_ids=beam.token_ids,
                    image_feature=image_feature,
                    previous_delta=beam.context_delta,
                    trace=trace,
                    trace_context={
                        "step": step_index + 1,
                        "beam_index": beam_index,
                        "beam_text": self.decoder.decode_full(beam.token_ids),
                        "generated_token_ids": [
                            int(value) for value in beam.generated_token_ids
                        ],
                    },
                )
                (
                    p_guided_constrained,
                    used_fallback,
                    constraint_diagnostics,
                ) = self.decoder.apply_guided_constraints(
                    step_result.p_guided_final,
                    beam.generated_token_ids,
                )
                if used_fallback:
                    p_final = p_guided_constrained
                else:
                    p_final = self.decoder.fuse(
                        p_guided_constrained,
                        step_result.p_original,
                        hard_suppressed_token_ids=(
                            constraint_diagnostics[
                                "hard_suppressed_token_ids"
                            ]
                        ),
                    )
                assert_probability_distribution(p_final, "P_final")

                positive_count = int(
                    torch.count_nonzero(p_final > 0).item()
                )
                expansion_count = min(
                    self.config.beam_size,
                    max(1, positive_count),
                )
                top_probabilities, top_ids = torch.topk(
                    p_final,
                    k=expansion_count,
                    dim=-1,
                )

                sibling_deltas = []
                expansion_rows = []
                for probability, token_id in zip(
                    top_probabilities[0],
                    top_ids[0],
                ):
                    token_value = int(token_id.item())
                    next_ids = torch.cat(
                        [beam.token_ids, token_id.view(1, 1)],
                        dim=1,
                    )
                    generated = (
                        *beam.generated_token_ids,
                        token_value,
                    )
                    stop_reason = self.decoder.stop_reason_for_token(
                        token_value,
                        used_fallback=used_fallback,
                    )
                    stopped = bool(stop_reason)

                    if (
                        step_index + 1 >= self.config.max_new_tokens
                        and not stopped
                    ):
                        stopped = True
                        stop_reason = "max_tokens"

                    child_delta = clone_context_delta(
                        step_result.context_delta
                    )
                    sibling_deltas.append(child_delta)
                    expansion_rows.append(
                        {
                            "token_id": token_value,
                            "token_text": self.models.gpt_tokenizer.decode(
                                [token_value],
                                skip_special_tokens=False,
                                clean_up_tokenization_spaces=False,
                            ),
                            "probability": float(probability.item()),
                            "next_full_text": self.decoder.decode_full(next_ids),
                            "stopped": stopped,
                            "stop_reason": stop_reason,
                        }
                    )
                    candidates.append(
                        BeamState(
                            token_ids=next_ids,
                            generated_token_ids=generated,
                            accumulated_logprob=(
                                beam.accumulated_logprob
                                + math.log(
                                    max(
                                        float(probability.item()),
                                        self.config.probability_eps,
                                    )
                                )
                            ),
                            stopped=stopped,
                            stop_reason=stop_reason,
                            context_delta=child_delta,
                        )
                    )

                if trace is not None:
                    trace.record(
                        "beam_distribution_and_expansion",
                        step=step_index + 1,
                        beam_index=beam_index,
                        beam_text=self.decoder.decode_full(beam.token_ids),
                        p_original=probability_diagnostics(
                            step_result.p_original,
                            self.models.gpt_tokenizer,
                            self.config.diagnostic_top_n,
                        ),
                        p_guided_final=probability_diagnostics(
                            step_result.p_guided_final,
                            self.models.gpt_tokenizer,
                            self.config.diagnostic_top_n,
                        ),
                        p_guided_constrained=probability_diagnostics(
                            p_guided_constrained,
                            self.models.gpt_tokenizer,
                            self.config.diagnostic_top_n,
                        ),
                        p_fused=probability_diagnostics(
                            p_final,
                            self.models.gpt_tokenizer,
                            self.config.diagnostic_top_n,
                        ),
                        constraints=constraint_diagnostics,
                        expansions=expansion_rows,
                    )

                non_null_deltas = [
                    delta
                    for delta in sibling_deltas
                    if delta is not None
                ]
                if len(non_null_deltas) > 1:
                    first_ptrs = {
                        delta[0][0].data_ptr()
                        for delta in non_null_deltas
                    }
                    if len(first_ptrs) != len(non_null_deltas):
                        raise AssertionError(
                            "Sibling beams unexpectedly share context delta storage."
                        )

            beams = sorted(
                candidates,
                key=lambda state: state.normalized_score(
                    self.config.beam_length_penalty
                ),
                reverse=True,
            )[: self.config.beam_size]

            if trace is not None:
                trace.record(
                    "beam_step_completed",
                    step=step_index + 1,
                    retained_beams=[
                        {
                            "text": self.decoder.decode_full(beam.token_ids),
                            "generated_token_ids": [
                                int(value) for value in beam.generated_token_ids
                            ],
                            "accumulated_logprob": float(
                                beam.accumulated_logprob
                            ),
                            "normalized_score": float(
                                beam.normalized_score(
                                    self.config.beam_length_penalty
                                )
                            ),
                            "stopped": bool(beam.stopped),
                            "stop_reason": beam.stop_reason,
                        }
                        for beam in beams
                    ],
                )

            if self.config.debug:
                print(
                    f"beam step {step_index + 1}:",
                    [
                        {
                            "text": self.decoder.decode_full(beam.token_ids),
                            "score": beam.accumulated_logprob,
                            "normalized_score": beam.normalized_score(
                                self.config.beam_length_penalty
                            ),
                            "stopped": beam.stopped,
                            "reason": beam.stop_reason,
                        }
                        for beam in beams
                    ],
                )
                print("P_final sum:", float(p_final.sum().item()))

            if beams and all(beam.stopped for beam in beams):
                break

        if loop_iterations > self.config.max_new_tokens:
            raise AssertionError("Generation loop exceeded max_new_tokens.")
        if not beams:
            raise RuntimeError("Beam search produced no beams.")

        for beam in beams:
            if not beam.stopped:
                beam.stopped = True
                beam.stop_reason = "max_tokens"
            if beam.stop_reason not in self.decoder.VALID_STOP_REASONS:
                raise AssertionError(
                    f"Invalid stop reason: {beam.stop_reason!r}"
                )
            if len(beam.generated_token_ids) > self.config.max_new_tokens:
                raise AssertionError("Generated token limit was exceeded.")
            if beam.context_delta is not None:
                for delta_key, delta_value in beam.context_delta:
                    for delta_tensor in (delta_key, delta_value):
                        if (
                            delta_tensor.requires_grad
                            or delta_tensor.grad_fn is not None
                        ):
                            raise AssertionError(
                                "A beam retained a computation graph."
                            )

        full_candidate_texts = [
            self.decoder.decode_full(beam.token_ids)
            for beam in beams
        ]
        candidate_captions = [
            self.decoder.caption_from_full(
                full_text=full_candidate_texts[index],
                prompt_text=prompt_text,
                generated_token_ids=beam.generated_token_ids,
            )
            for index, beam in enumerate(beams)
        ]
        if self.config.final_rerank_text_mode == "caption_only":
            rerank_candidate_texts = candidate_captions
        elif self.config.final_rerank_text_mode == "prompt_plus_caption":
            rerank_candidate_texts = full_candidate_texts
        else:
            raise AssertionError(
                f"Unsupported final rerank text mode: "
                f"{self.config.final_rerank_text_mode!r}."
            )
        normalized_beam_scores = [
            float(
                beam.normalized_score(
                    self.config.beam_length_penalty
                )
            )
            for beam in beams
        ]
        (
            best_index,
            similarities,
            normalized_beam_scores,
            rerank_scores,
            clip_only_index,
            beam_only_index,
        ) = self.guidance.rerank(
            image_feature=image_feature,
            candidate_texts=rerank_candidate_texts,
            normalized_beam_scores=normalized_beam_scores,
        )
        best_beam = beams[best_index]
        caption = candidate_captions[best_index]
        if not caption:
            raise AssertionError("Generated caption is empty.")

        if trace is not None:
            trace.record(
                "final_clip_rerank",
                rerank_mode=self.config.final_rerank_mode,
                rerank_text_mode=self.config.final_rerank_text_mode,
                clip_weight=float(
                    self.config.final_rerank_clip_weight
                ),
                beam_weight=float(
                    self.config.final_rerank_beam_weight
                ),
                clip_only_beam_index=clip_only_index,
                beam_only_beam_index=beam_only_index,
                selected_differs_from_clip_only=(
                    best_index != clip_only_index
                ),
                selected_differs_from_beam_only=(
                    best_index != beam_only_index
                ),
                candidates=[
                    {
                        "beam_index": index,
                        "full_text": full_candidate_texts[index],
                        "caption": candidate_captions[index],
                        "rerank_text": rerank_candidate_texts[index],
                        "clip_similarity": float(similarities[index]),
                        "weighted_clip_component": float(
                            self.config.final_rerank_clip_weight
                            * similarities[index]
                        ),
                        "weighted_beam_component": float(
                            (
                                self.config.final_rerank_beam_weight
                                * normalized_beam_scores[index]
                            )
                            if self.config.final_rerank_mode
                            == "clip_beam"
                            else 0.0
                        ),
                        "rerank_score": float(rerank_scores[index]),
                        "accumulated_logprob": float(
                            beams[index].accumulated_logprob
                        ),
                        "normalized_beam_score": float(
                            normalized_beam_scores[index]
                        ),
                        "stop_reason": beams[index].stop_reason,
                        "selected": index == best_index,
                    }
                    for index in range(len(beams))
                ],
                selected_beam_index=best_index,
                selected_caption=caption,
            )

        if self.config.debug:
            print("Final beam full texts:", full_candidate_texts)
            print("Final candidate captions:", candidate_captions)
            print(
                f"Final rerank texts ({self.config.final_rerank_text_mode}):",
                rerank_candidate_texts,
            )
            print("Final CLIP similarities:", similarities)
            print("Final normalized beam scores:", normalized_beam_scores)
            print(
                f"Final rerank scores ({self.config.final_rerank_mode}):",
                rerank_scores,
            )
            print(
                "CLIP-only / beam-only / selected indices:",
                clip_only_index,
                beam_only_index,
                best_index,
            )
            print("Selected caption:", caption)

        payload = {
            "caption": caption,
            "generated_token_ids": [
                int(value)
                for value in best_beam.generated_token_ids
            ],
            "stop_reason": best_beam.stop_reason,
            "final_clip_similarity": float(similarities[best_index]),
        }
        del beams, candidates
        return payload

## 13. Public ZeroCapCaptioner interface

In [ ]:
class ZeroCapCaptioner:
    def __init__(self, config, models=None):
        self.config = config
        if models is None:
            self.models = ZeroCapModels(config)
        else:
            compatibility_fields = (
                "gpt_model",
                "clip_model",
                "device",
                "model_dtype",
            )
            mismatches = {
                field_name: (
                    getattr(models.config, field_name),
                    getattr(config, field_name),
                )
                for field_name in compatibility_fields
                if getattr(models.config, field_name)
                != getattr(config, field_name)
            }
            if mismatches:
                raise ValueError(
                    f"Shared model bundle is incompatible: {mismatches}"
                )
            self.models = models
            self.models._assert_models()
        self.image_encoder = ImageEncoder(self.models, config)
        self.guidance = CLIPGuidance(self.models, config)
        self.context_optimizer = ContextOptimizer(
            self.models,
            self.guidance,
            config,
        )
        self.decoder = ZeroCapDecoder(self.models, config)
        self.generator = ZeroCapGenerator(
            self.models,
            self.guidance,
            self.context_optimizer,
            self.decoder,
            config,
        )
        self.assert_effective_config()

    def assert_effective_config(self):
        components = {
            "captioner": self,
            "image_encoder": self.image_encoder,
            "guidance": self.guidance,
            "context_optimizer": self.context_optimizer,
            "decoder": self.decoder,
            "generator": self.generator,
        }
        stale = [
            name
            for name, component in components.items()
            if component.config is not self.config
        ]
        if stale:
            raise AssertionError(
                f"Components hold a stale config object: {stale}"
            )
        return {
            "candidate": self.config.val_tune_candidate,
            "case_reason": self.config.val_tune_case_reason,
            "config_hash": self.config.config_hash,
            "fluency_weight": float(self.context_optimizer.config.fluency_weight),
            "fusion_factor": float(self.decoder.config.fusion_factor),
            "clip_temperature": float(self.guidance.config.clip_temperature),
            "prepend_bos_token": bool(self.generator.config.prepend_bos_token),
            "final_rerank_text_mode": (
                self.generator.config.final_rerank_text_mode
            ),
            "algorithm_revision": self.config.algorithm_revision,
            "all_component_configs_current": True,
        }

    def generate_caption(self, image, image_id):
        trace = (
            GenerationTrace(self.config, image_id, image)
            if self.config.save_diagnostics
            else None
        )
        image_feature = None
        payload = None

        try:
            gpt_snapshot = parameter_snapshot(self.models.gpt_model)
            clip_snapshot = parameter_snapshot(self.models.clip_model)
            assert_model_parameter_grads_clear(self.models.gpt_model, "GPT-2")
            assert_model_parameter_grads_clear(self.models.clip_model, "CLIP")

            torch.cuda.synchronize()
            end_to_end_start = time.perf_counter()
            image_encoding_start = time.perf_counter()
            image_feature = self.image_encoder.encode(
                image=image,
                image_id=image_id,
            )
            torch.cuda.synchronize()
            image_encoding_time_sec = (
                time.perf_counter() - image_encoding_start
            )

            if trace is not None:
                trace.record(
                    "image_encoded",
                    feature_shape=list(image_feature.shape),
                    feature_dtype=str(image_feature.dtype),
                    feature_device=str(image_feature.device),
                    feature_norm=float(image_feature.norm().item()),
                    feature_min=float(image_feature.min().item()),
                    feature_max=float(image_feature.max().item()),
                    image_encoding_time_sec=float(image_encoding_time_sec),
                )

            generation_start = time.perf_counter()
            payload = self.generator.generate(
                image_feature,
                trace=trace,
            )
            torch.cuda.synchronize()
            generation_time_sec = time.perf_counter() - generation_start
            end_to_end_time_sec = time.perf_counter() - end_to_end_start

            result = GenerationResult(
                image_id=str(image_id),
                caption=str(payload["caption"]),
                generated_token_ids=list(payload["generated_token_ids"]),
                num_generated_tokens=len(payload["generated_token_ids"]),
                stop_reason=str(payload["stop_reason"]),
                generation_time_sec=float(generation_time_sec),
                end_to_end_time_sec=float(end_to_end_time_sec),
                final_clip_similarity=float(payload["final_clip_similarity"]),
                gpt_model=self.config.gpt_model,
                clip_model=self.config.clip_model,
                config_hash=self.config.config_hash,
                git_commit=self.config.git_commit,
            )

            if not result.caption.strip():
                raise AssertionError("Caption is empty.")
            if result.num_generated_tokens != len(result.generated_token_ids):
                raise AssertionError("Generated token count mismatch.")
            if result.num_generated_tokens > self.config.max_new_tokens:
                raise AssertionError("Generated token count exceeds configured limit.")
            if result.stop_reason not in self.decoder.VALID_STOP_REASONS:
                raise AssertionError("Invalid generation stop reason.")

            assert_parameter_snapshot_unchanged(
                self.models.gpt_model,
                gpt_snapshot,
                "GPT-2",
            )
            assert_parameter_snapshot_unchanged(
                self.models.clip_model,
                clip_snapshot,
                "CLIP",
            )
            assert_model_parameter_grads_clear(self.models.gpt_model, "GPT-2")
            assert_model_parameter_grads_clear(self.models.clip_model, "CLIP")

            public_result = result.to_dict()
            if any(torch.is_tensor(value) for value in public_result.values()):
                raise AssertionError("Public result contains a GPU tensor.")

            if trace is not None:
                trace.record(
                    "generation_finished",
                    generation_time_sec=float(generation_time_sec),
                    end_to_end_time_sec=float(end_to_end_time_sec),
                    cuda_memory={
                        "allocated_mb": float(
                            torch.cuda.memory_allocated() / (1024 ** 2)
                        ),
                        "reserved_mb": float(
                            torch.cuda.memory_reserved() / (1024 ** 2)
                        ),
                        "peak_allocated_mb": float(
                            torch.cuda.max_memory_allocated() / (1024 ** 2)
                        ),
                    },
                )
                trace.finish(public_result)
                trace_path = trace.save()
                print("Diagnostic trace saved:", trace_path)

        except Exception as error:
            if trace is not None:
                trace.fail(error)
                trace_path = trace.save()
                print("FAILED diagnostic trace saved:", trace_path)
            if image_feature is not None:
                del image_feature
            if payload is not None:
                del payload
            gc.collect()
            raise

        del image_feature, payload, result
        gc.collect()
        return public_result


captioner = ZeroCapCaptioner(config)
print("ZeroCapCaptioner ready:", config.gpt_model, "+", config.clip_model)

## 14. Smoke mode

Một fixed VAL image theo seed 42, debug profile tối đa ba token. Notebook hiển thị ảnh đầu vào và lưu trace JSON riêng cho từng ảnh trong `RUN_DIR/diagnostics/`: cache/delta, loss và gradient từng inner iteration, Top-K GPT/CLIP, original-guided-fused-constrained distributions, beam pruning và final reranking (CLIP-only, beam-only, joint score). Nếu generation lỗi, trace đã thu thập cùng traceback vẫn được lưu. Reference captions vẫn chưa được mở.

In [ ]:
SMOKE_RESULT = None

if config.run_mode == "smoke":
    image_id = SMOKE_VAL_IMAGE_ID
    if image_id in store.completed_ids():
        SMOKE_RESULT = store.predictions[image_id]
        print("Resume: smoke image already completed:", image_id)
    else:
        image = data.load_image(image_id)
        if config.display_input_images:
            print("Smoke input image:", image_id, image.size, image.mode)
            display(image)
        SMOKE_RESULT = captioner.generate_caption(
            image=image,
            image_id=image_id,
        )
        store.save_prediction(SMOKE_RESULT)
        del image

    assert not data.references_loaded
    assert SMOKE_RESULT["num_generated_tokens"] <= 3
    assert SMOKE_RESULT["caption"].strip()
    print(json.dumps(SMOKE_RESULT, ensure_ascii=False, indent=2))
    print("SMOKE STATIC/RUNTIME ASSERTIONS: PASS")
else:
    print("Smoke cell skipped for RUN_MODE =", config.run_mode)

## 15. Final-profile VAL preview and benchmark

Trong benchmark mode, warm-up đầu tiên chính là fixed smoke VAL image nhưng chạy final profile. Tổng cộng hai warm-up ảnh không tính timing. Sau đó benchmark 5 hoặc 10 ảnh khác, sync CUDA, reset peak memory từng ảnh và lưu sau từng ảnh. Khi diagnostic trace đang bật, timing chỉ dùng để tìm nguyên nhân chất lượng/tốc độ và notebook không ghi đè canonical fixed TEST IDs; hãy tắt trace rồi benchmark lại sau khi chốt bản sửa để sizing TEST. Không chạy TEST trong cell này.

In [ ]:
BENCHMARK_SUMMARY = None

if config.run_mode == "benchmark":
    print("Final-profile preview uses smoke VAL image:", VAL_WARMUP_IDS[0])
    for warmup_index, image_id in enumerate(VAL_WARMUP_IDS, start=1):
        image = data.load_image(image_id)
        if config.display_input_images:
            print("Warm-up input image:", image_id, image.size, image.mode)
            display(image)
        torch.cuda.synchronize()
        warmup_result = captioner.generate_caption(
            image=image,
            image_id=image_id,
        )
        torch.cuda.synchronize()
        print(
            f"Warm-up {warmup_index}/{len(VAL_WARMUP_IDS)}:",
            image_id,
            "→",
            warmup_result["caption"],
        )
        del image, warmup_result
        gc.collect()

    for benchmark_index, image_id in enumerate(
        VAL_BENCHMARK_IDS,
        start=1,
    ):
        if image_id in store.completed_ids():
            print("Resume benchmark:", image_id)
            continue

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        end_to_end_start = time.perf_counter()
        image = data.load_image(image_id)
        result = captioner.generate_caption(
            image=image,
            image_id=image_id,
        )
        torch.cuda.synchronize()
        measured_end_to_end = time.perf_counter() - end_to_end_start
        if config.display_input_images:
            print("Benchmark input image:", image_id, image.size, image.mode)
            display(image)
        peak_vram_mb = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )
        result["end_to_end_time_sec"] = float(measured_end_to_end)
        result["peak_vram_mb"] = float(peak_vram_mb)
        store.save_prediction(result)
        print(
            f"Benchmark {benchmark_index}/{len(VAL_BENCHMARK_IDS)}:",
            image_id,
            f"{measured_end_to_end:.2f}s",
            f"{peak_vram_mb:.1f} MiB",
            "→",
            result["caption"],
        )
        del image, result
        gc.collect()

    benchmark_rows = [
        store.predictions[image_id]
        for image_id in VAL_BENCHMARK_IDS
        if image_id in store.predictions
    ]
    if len(benchmark_rows) != len(VAL_BENCHMARK_IDS):
        raise RuntimeError("Benchmark predictions are incomplete.")

    times = np.asarray(
        [row["end_to_end_time_sec"] for row in benchmark_rows],
        dtype=np.float64,
    )
    generation_times = np.asarray(
        [row["generation_time_sec"] for row in benchmark_rows],
        dtype=np.float64,
    )
    token_counts = np.asarray(
        [row["num_generated_tokens"] for row in benchmark_rows],
        dtype=np.float64,
    )
    peaks = np.asarray(
        [row["peak_vram_mb"] for row in benchmark_rows],
        dtype=np.float64,
    )
    p95_time = float(np.percentile(times, 95))
    average_time = float(times.mean())
    safe_time_per_image = max(
        p95_time,
        average_time * 1.2,
    )
    n_test = min(
        len(TEST_IDS),
        int(
            config.time_budget_hours * 3600
            / safe_time_per_image
        ),
    )
    fixed_test_ids = deterministic_sample(
        TEST_IDS,
        n_test,
        config.seed,
    )
    data.assert_ids_exist(fixed_test_ids)

    canonical_fixed_path = OUTPUT_ROOT / "fixed_test_image_ids.json"
    if not config.save_diagnostics:
        PredictionStore._atomic_json(
            canonical_fixed_path,
            fixed_test_ids,
        )
    store.save_artifact_json(
        "fixed_test_image_ids.json",
        fixed_test_ids,
    )
    fixed_metadata = {
        "seed": config.seed,
        "n_test": n_test,
        "total_test_images": len(TEST_IDS),
        "safe_time_per_image_sec": safe_time_per_image,
        "time_budget_hours": config.time_budget_hours,
        "source_run_dir": str(RUN_DIR),
        "git_commit": config.git_commit,
        "split_manifest_sha256": config.split_manifest_sha256,
    }
    if not config.save_diagnostics:
        PredictionStore._atomic_json(
            OUTPUT_ROOT / "fixed_test_image_ids_metadata.json",
            fixed_metadata,
        )
    store.save_artifact_json(
        "fixed_test_image_ids_metadata.json",
        fixed_metadata,
    )

    BENCHMARK_SUMMARY = {
        "count": len(times),
        "mean_sec": average_time,
        "median_sec": float(np.median(times)),
        "p95_sec": p95_time,
        "min_sec": float(times.min()),
        "max_sec": float(times.max()),
        "mean_generation_sec": float(generation_times.mean()),
        "mean_time_per_token_sec": float(
            np.mean(generation_times / np.maximum(token_counts, 1))
        ),
        "peak_vram_mb": float(peaks.max()),
        "safe_time_per_image_sec": safe_time_per_image,
        "estimated_n_test": n_test,
        "total_test_images": len(TEST_IDS),
        "diagnostic_trace_enabled": config.save_diagnostics,
        "timing_valid_for_test_sizing": not config.save_diagnostics,
    }
    store.save_artifact_json(
        "benchmark_summary.json",
        BENCHMARK_SUMMARY,
    )
    assert not data.references_loaded
    print(json.dumps(BENCHMARK_SUMMARY, indent=2))
    if config.save_diagnostics:
        print(
            "Diagnostic timing only; canonical fixed TEST IDs were not overwritten."
        )
    else:
        print("Saved reusable fixed TEST IDs:", canonical_fixed_path)
    print("Benchmark mode does not run final TEST.")
else:
    print("Benchmark cell skipped for RUN_MODE =", config.run_mode)

## 16. TEST smoke/final execution with per-image resume

Cả hai TEST modes đều yêu cầu canonical fixed_test_image_ids.json do benchmark tạo. test_smoke lấy đúng năm ID đầu; final_test chỉ chạy đúng danh sách trong file, không tự mở rộng lên 809. Reference captions vẫn chưa được đọc.

In [ ]:
REQUESTED_EVAL_IDS = []

if config.run_mode in {"test_smoke", "final_test"}:
    fixed_test_path = OUTPUT_ROOT / "fixed_test_image_ids.json"
    if not fixed_test_path.is_file():
        raise FileNotFoundError(
            "fixed_test_image_ids.json is missing. Run benchmark mode first; "
            "do not invent or tune TEST IDs from metrics."
        )
    with fixed_test_path.open("r", encoding="utf-8") as handle:
        fixed_test_ids = json.load(handle)
    if (
        not isinstance(fixed_test_ids, list)
        or not fixed_test_ids
        or len(fixed_test_ids) != len(set(fixed_test_ids))
        or not all(isinstance(value, str) for value in fixed_test_ids)
    ):
        raise RuntimeError("fixed_test_image_ids.json is invalid.")
    if not set(fixed_test_ids).issubset(set(TEST_IDS)):
        raise RuntimeError("Fixed TEST file contains IDs outside the TEST split.")

    if config.run_mode == "test_smoke":
        if len(fixed_test_ids) < config.test_smoke_images:
            raise RuntimeError(
                "Fixed TEST file contains fewer than five IDs."
            )
        REQUESTED_EVAL_IDS = fixed_test_ids[: config.test_smoke_images]
    else:
        REQUESTED_EVAL_IDS = fixed_test_ids

    data.assert_ids_exist(REQUESTED_EVAL_IDS)
    assert not data.references_loaded

    already_complete = store.completed_ids()
    remaining_ids = [
        image_id
        for image_id in REQUESTED_EVAL_IDS
        if image_id not in already_complete
    ]
    resumed_count = len(REQUESTED_EVAL_IDS) - len(remaining_ids)
    print(
        f"Requested={len(REQUESTED_EVAL_IDS)}, "
        f"resumed={resumed_count}, remaining={len(remaining_ids)}"
    )

    loop_start = time.perf_counter()
    newly_completed = 0
    for image_id in remaining_ids:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        end_to_end_start = time.perf_counter()
        image = data.load_image(image_id)
        result = captioner.generate_caption(
            image=image,
            image_id=image_id,
        )
        torch.cuda.synchronize()
        result["end_to_end_time_sec"] = float(
            time.perf_counter() - end_to_end_start
        )
        result["peak_vram_mb"] = float(
            torch.cuda.max_memory_allocated() / (1024 ** 2)
        )
        store.save_prediction(result)
        newly_completed += 1

        elapsed = time.perf_counter() - loop_start
        average_new = elapsed / newly_completed
        left = len(remaining_ids) - newly_completed
        eta_seconds = average_new * left
        print(
            f"[{resumed_count + newly_completed}/"
            f"{len(REQUESTED_EVAL_IDS)}] {image_id} "
            f"| {result['end_to_end_time_sec']:.2f}s "
            f"| ETA {eta_seconds / 60:.1f} min "
            f"| {result['caption']}"
        )
        del image, result
        gc.collect()

    missing_after_run = [
        image_id
        for image_id in REQUESTED_EVAL_IDS
        if image_id not in store.completed_ids()
    ]
    if missing_after_run:
        raise RuntimeError(
            f"Predictions remain incomplete: {missing_after_run[:5]}"
        )
    assert not data.references_loaded
    print(config.run_mode, "prediction/resume pipeline: PASS")
else:
    print("TEST execution skipped for RUN_MODE =", config.run_mode)

## 17. Optional evaluator — references enter only here

Evaluator chạy sau khi toàn bộ requested predictions tồn tại. BLEU, ROUGE-L và CIDEr được tính trước; METEOR chạy trong process riêng với hard timeout để Java scorer không thể treo cả notebook. Mỗi metric được cô lập lỗi và predictions đã checkpoint không bị xóa. CLIPScore được báo riêng với cảnh báo thiên vị vì dùng cùng CLIP backbone với guidance.

In [ ]:
class Flickr8kEvaluator:
    _meteor_runtime_disabled_reason = None

    def __init__(self, config):
        self.config = config

    @staticmethod
    def _compute_metric(name, function, metrics, errors):
        try:
            function()
        except Exception as exc:
            errors[name] = f"{type(exc).__name__}: {exc}"

    def evaluate(
        self,
        data,
        image_ids,
        predictions,
        split="test",
    ):
        if split not in {"val", "test"}:
            raise ValueError(f"Unsupported evaluation split: {split}")
        if not image_ids:
            raise ValueError("No image IDs were supplied for evaluation.")
        missing = [
            image_id
            for image_id in image_ids
            if image_id not in predictions
        ]
        if missing:
            raise RuntimeError(
                f"Evaluation cannot run before predictions finish: {missing[:5]}"
            )

        references = data.load_references(split, image_ids)
        gts = {
            image_id: [
                postprocess_caption(reference, self.config.prompt)
                for reference in references[image_id]
            ]
            for image_id in image_ids
        }
        res = {
            image_id: [
                postprocess_caption(
                    predictions[image_id]["caption"],
                    self.config.prompt,
                )
            ]
            for image_id in image_ids
        }
        if not all(len(gts[image_id]) == 5 for image_id in image_ids):
            raise AssertionError("Evaluator did not receive five references/image.")

        metrics = {
            "num_images": len(image_ids),
            "evaluation_split": split,
            "run_mode": self.config.run_mode,
            "config_hash": self.config.config_hash,
            "git_commit": self.config.git_commit,
            "clipscore_bias_note": (
                "CLIPScore is supplementary and may favor ZeroCap because "
                "generation guidance uses the same CLIP backbone."
            ),
        }
        errors = {}

        from pycocoevalcap.bleu.bleu import Bleu
        from pycocoevalcap.cider.cider import Cider
        from pycocoevalcap.rouge.rouge import Rouge

        def compute_bleu():
            scores, _ = Bleu(4).compute_score(gts, res)
            metrics["BLEU-1"] = float(scores[0])
            metrics["BLEU-4"] = float(scores[3])

        def compute_meteor():
            meteor_payload = json.dumps(
                {"gts": gts, "res": res},
                ensure_ascii=False,
            )
            meteor_script = r'''
import json
import sys
from pycocoevalcap.meteor.meteor import Meteor

payload = json.loads(sys.stdin.read())
scorer = Meteor()
try:
    score, _ = scorer.compute_score(payload["gts"], payload["res"])
    print(
        "METEOR_RESULT=" + json.dumps({"score": float(score)}),
        flush=True,
    )
finally:
    if hasattr(scorer, "close"):
        scorer.close()
'''
            process = subprocess.Popen(
                [sys.executable, "-c", meteor_script],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                start_new_session=True,
            )
            try:
                stdout, stderr = process.communicate(
                    input=meteor_payload,
                    timeout=float(self.config.metric_timeout_sec),
                )
            except subprocess.TimeoutExpired as exc:
                try:
                    if hasattr(os, "killpg"):
                        os.killpg(process.pid, signal.SIGKILL)
                    else:
                        process.kill()
                except ProcessLookupError:
                    pass
                process.communicate()
                raise TimeoutError(
                    "METEOR exceeded "
                    f"{self.config.metric_timeout_sec:.0f}s and was killed."
                ) from exc
            if process.returncode != 0:
                raise RuntimeError(
                    "Isolated METEOR process failed: "
                    + stderr.strip()[-1000:]
                )
            result_lines = [
                line
                for line in stdout.splitlines()
                if line.startswith("METEOR_RESULT=")
            ]
            if len(result_lines) != 1:
                raise RuntimeError(
                    "Isolated METEOR process returned no unique result."
                )
            meteor_result = json.loads(
                result_lines[0].split("=", 1)[1]
            )
            metrics["METEOR"] = float(meteor_result["score"])

        def compute_rouge():
            score, _ = Rouge().compute_score(gts, res)
            metrics["ROUGE-L"] = float(score)

        def compute_cider():
            score, _ = Cider().compute_score(gts, res)
            metrics["CIDEr"] = float(score)

        self._compute_metric("BLEU", compute_bleu, metrics, errors)
        self._compute_metric("ROUGE-L", compute_rouge, metrics, errors)
        self._compute_metric("CIDEr", compute_cider, metrics, errors)
        if type(self)._meteor_runtime_disabled_reason is not None:
            errors["METEOR"] = (
                "skipped after an earlier timeout in this runtime: "
                + type(self)._meteor_runtime_disabled_reason
            )
        else:
            self._compute_metric(
                "METEOR",
                compute_meteor,
                metrics,
                errors,
            )
            meteor_error = errors.get("METEOR", "")
            if meteor_error.startswith("TimeoutError:"):
                type(self)._meteor_runtime_disabled_reason = meteor_error

        if self.config.run_heavy_metrics:
            def compute_spice():
                from pycocoevalcap.spice.spice import Spice

                score, _ = Spice().compute_score(gts, res)
                metrics["SPICE"] = float(score)

            self._compute_metric("SPICE", compute_spice, metrics, errors)
        else:
            metrics["SPICE"] = "skipped (RUN_HEAVY_METRICS=False)"

        raw_similarities = [
            float(predictions[image_id]["final_clip_similarity"])
            for image_id in image_ids
        ]
        metrics["CLIPScore_raw_cosine"] = float(
            np.mean(raw_similarities)
        )
        metrics["CLIPScore"] = float(
            np.mean([
                2.5 * max(value, 0.0)
                for value in raw_similarities
            ])
        )
        metrics["errors"] = errors
        return metrics


EVALUATION_RESULT = None
if config.run_mode in {"test_smoke", "final_test"}:
    evaluator = Flickr8kEvaluator(config)
    try:
        EVALUATION_RESULT = evaluator.evaluate(
            data=data,
            image_ids=REQUESTED_EVAL_IDS,
            predictions=store.predictions,
        )
    except Exception as exc:
        EVALUATION_RESULT = {
            "num_images": len(REQUESTED_EVAL_IDS),
            "run_mode": config.run_mode,
            "config_hash": config.config_hash,
            "git_commit": config.git_commit,
            "fatal_evaluation_error": (
                f"{type(exc).__name__}: {exc}"
            ),
            "predictions_preserved": True,
        }
    finally:
        store.save_metrics(EVALUATION_RESULT)

    print(json.dumps(EVALUATION_RESULT, ensure_ascii=False, indent=2))
    print("Predictions remain at:", store.predictions_path)
else:
    print("Evaluation skipped for RUN_MODE =", config.run_mode)

## 18. VAL hyperparameter sensitivity — `RUN_MODE="val_tune"`

Mode này không training và không thay đổi GPT-2/CLIP weights. Nó chạy năm case one-factor-at-a-time trên cùng VAL_BENCHMARK_IDS: baseline, hai mức giảm fluency/KL, CLIP temperature mềm hơn và fusion 0.95. Mỗi candidate có lý do thử nghiệm, config hash, run directory, prediction store và component config riêng; model frozen được dùng chung. Diagnostics được bật để giải thích thay đổi token/loss/probability. Tất cả prediction được sinh xong trước khi evaluator đọc VAL references; METEOR chạy cô lập với timeout.

In [ ]:
VAL_TUNE_SUMMARY = None


def load_val_tune_trace_probe(candidate_config, image_id):
    trace_path = (
        Path(candidate_config.run_dir)
        / "diagnostics"
        / f"{Path(image_id).name}.json"
    )
    if not trace_path.is_file():
        raise FileNotFoundError(
            f"Missing VAL-tune diagnostic trace: {trace_path}"
        )
    with trace_path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("status") != "success":
        raise RuntimeError(
            f"VAL-tune trace is not successful: {trace_path}"
        )

    cache_equivalence_event = next(
        (
            event
            for event in payload["events"]
            if event.get("event") == "zero_delta_cache_equivalence"
            and event.get("context", {}).get("step") == 1
            and event.get("context", {}).get("beam_index") == 0
        ),
        None,
    )
    post_update_event = next(
        (
            event
            for event in payload["events"]
            if event.get("event")
            == "inner_optimization_post_update_evaluation"
            and event.get("context", {}).get("step") == 1
            and event.get("context", {}).get("beam_index") == 0
        ),
        None,
    )
    distribution_event = next(
        (
            event
            for event in payload["events"]
            if event.get("event") == "beam_distribution_and_expansion"
            and event.get("step") == 1
            and event.get("beam_index") == 0
        ),
        None,
    )
    if (
        cache_equivalence_event is None
        or post_update_event is None
        or distribution_event is None
    ):
        raise RuntimeError(
            f"Trace lacks the first-step sensitivity events: {trace_path}"
        )

    diagnostics = post_update_event["diagnostics"]
    p_fused = distribution_event["p_fused"]
    top_token = p_fused["top_tokens"][0]
    return {
        "trace_path": str(trace_path),
        "cache_max_abs_error": float(
            cache_equivalence_event["max_abs_error"]
        ),
        "cache_mean_abs_error": float(
            cache_equivalence_event["mean_abs_error"]
        ),
        "cache_same_top_token": bool(
            cache_equivalence_event["same_top_token"]
        ),
        "loss_clip": float(diagnostics["loss_clip"]),
        "loss_fluency": float(diagnostics["loss_fluency"]),
        "weighted_fluency_loss": float(
            candidate_config.fluency_weight
            * diagnostics["loss_fluency"]
        ),
        "loss_total": float(diagnostics["loss_total"]),
        "delta_global_norm": float(
            diagnostics["delta_after"]["global_norm"]
        ),
        "p_fused_entropy": float(p_fused["entropy"]),
        "p_fused_top_token_id": int(top_token["token_id"]),
        "p_fused_top_token_text": str(top_token["token_text"]),
        "p_fused_top_probability": float(top_token["probability"]),
    }


if config.run_mode == "val_tune":
    grid = ZeroCapRunConfig.val_tune_grid()
    candidate_configs = [
        (
            candidate_name,
            config.for_val_tune_candidate(
                candidate_name,
                overrides,
            ),
        )
        for candidate_name, overrides in grid
    ]
    candidate_hashes = [
        candidate_config.config_hash
        for _, candidate_config in candidate_configs
    ]
    candidate_run_dirs = [
        candidate_config.run_dir
        for _, candidate_config in candidate_configs
    ]
    if len(candidate_hashes) != len(set(candidate_hashes)):
        raise AssertionError("VAL-tune candidate config hashes are not unique.")
    if len(candidate_run_dirs) != len(set(candidate_run_dirs)):
        raise AssertionError("VAL-tune candidate run directories are not unique.")

    candidate_stores = {}
    effective_configs = {}
    trace_probes = {}
    suite_gpt_snapshot = parameter_snapshot(captioner.models.gpt_model)
    suite_clip_snapshot = parameter_snapshot(captioner.models.clip_model)

    print(
        "VAL-tune diagnostic grid:",
        [
            {
                "candidate": name,
                "fluency_weight": candidate_config.fluency_weight,
                "fusion_factor": candidate_config.fusion_factor,
                "clip_temperature": candidate_config.clip_temperature,
                "case_reason": candidate_config.val_tune_case_reason,
                "config_hash": candidate_config.config_hash,
            }
            for name, candidate_config in candidate_configs
        ],
    )
    print(
        "Model bundle is loaded once and remains frozen; "
        "candidate inference components are rebuilt per config."
    )

    for candidate_index, (
        candidate_name,
        candidate_config,
    ) in enumerate(candidate_configs, start=1):
        seed_everything(candidate_config.seed)
        candidate_store = PredictionStore(
            config=candidate_config,
            library_versions=LIBRARY_VERSIONS,
            gpu_name=GPU_NAME,
        )
        candidate_captioner = ZeroCapCaptioner(
            candidate_config,
            models=captioner.models,
        )
        effective = candidate_captioner.assert_effective_config()
        expected_overrides = dict(grid[candidate_index - 1][1])
        if not math.isclose(
            effective["fluency_weight"],
            float(expected_overrides["fluency_weight"]),
            rel_tol=0.0,
            abs_tol=0.0,
        ):
            raise AssertionError("ContextOptimizer received stale fluency_weight.")
        if not math.isclose(
            effective["fusion_factor"],
            float(expected_overrides["fusion_factor"]),
            rel_tol=0.0,
            abs_tol=0.0,
        ):
            raise AssertionError("ZeroCapDecoder received stale fusion_factor.")
        if not math.isclose(
            effective["clip_temperature"],
            float(expected_overrides["clip_temperature"]),
            rel_tol=0.0,
            abs_tol=0.0,
        ):
            raise AssertionError("CLIPGuidance received stale clip_temperature.")
        if effective["case_reason"] != expected_overrides["case_reason"]:
            raise AssertionError("VAL-tune case reason was not preserved.")
        if not effective["prepend_bos_token"]:
            raise AssertionError("VAL-tune candidate did not enable GPT-2 BOS.")
        if effective["final_rerank_text_mode"] != "caption_only":
            raise AssertionError("VAL-tune rerank must use caption-only text.")

        print(
            f"[Candidate {candidate_index}/{len(candidate_configs)}]",
            json.dumps(effective, ensure_ascii=False),
        )
        for image_index, image_id in enumerate(
            VAL_BENCHMARK_IDS,
            start=1,
        ):
            if image_id in candidate_store.completed_ids():
                result = candidate_store.predictions[image_id]
                print(
                    f"  Resume {image_index}/{len(VAL_BENCHMARK_IDS)}:",
                    image_id,
                    "→",
                    result["caption"],
                )
                continue

            image = data.load_image(image_id)
            result = candidate_captioner.generate_caption(
                image=image,
                image_id=image_id,
            )
            result["val_tune_candidate"] = candidate_name
            result["fluency_weight"] = float(
                candidate_config.fluency_weight
            )
            result["fusion_factor"] = float(
                candidate_config.fusion_factor
            )
            result["clip_temperature"] = float(
                candidate_config.clip_temperature
            )
            result["case_reason"] = (
                candidate_config.val_tune_case_reason
            )
            candidate_store.save_prediction(result)
            print(
                f"  Generate {image_index}/{len(VAL_BENCHMARK_IDS)}:",
                image_id,
                "→",
                result["caption"],
            )
            del image, result
            gc.collect()

        completed = candidate_store.completed_ids()
        missing = [
            image_id
            for image_id in VAL_BENCHMARK_IDS
            if image_id not in completed
        ]
        if missing:
            raise RuntimeError(
                f"Candidate {candidate_name} is incomplete: {missing}"
            )

        candidate_stores[candidate_name] = candidate_store
        effective_configs[candidate_name] = effective
        trace_probes[candidate_name] = {
            image_id: load_val_tune_trace_probe(
                candidate_config,
                image_id,
            )
            for image_id in VAL_BENCHMARK_IDS
        }
        del candidate_captioner
        gc.collect()
        torch.cuda.empty_cache()

    assert_parameter_snapshot_unchanged(
        captioner.models.gpt_model,
        suite_gpt_snapshot,
        "GPT-2 across VAL-tune suite",
    )
    assert_parameter_snapshot_unchanged(
        captioner.models.clip_model,
        suite_clip_snapshot,
        "CLIP across VAL-tune suite",
    )
    assert_model_parameter_grads_clear(
        captioner.models.gpt_model,
        "GPT-2 across VAL-tune suite",
    )
    assert_model_parameter_grads_clear(
        captioner.models.clip_model,
        "CLIP across VAL-tune suite",
    )

    if data.references_loaded:
        raise AssertionError(
            "VAL references were loaded before all grid predictions finished."
        )

    comparisons = []
    for image_index, image_id in enumerate(
        VAL_BENCHMARK_IDS,
        start=1,
    ):
        rows = {
            candidate_name: {
                "caption": candidate_stores[
                    candidate_name
                ].predictions[image_id]["caption"],
                "generated_token_ids": candidate_stores[
                    candidate_name
                ].predictions[image_id]["generated_token_ids"],
                "probe": trace_probes[candidate_name][image_id],
            }
            for candidate_name, _ in candidate_configs
        }
        captions = [row["caption"] for row in rows.values()]
        token_sequences = [
            tuple(row["generated_token_ids"])
            for row in rows.values()
        ]
        probe_signatures = {
            (
                round(row["probe"]["loss_total"], 10),
                round(row["probe"]["delta_global_norm"], 10),
                round(row["probe"]["p_fused_entropy"], 10),
                row["probe"]["p_fused_top_token_id"],
                round(row["probe"]["p_fused_top_probability"], 10),
            )
            for row in rows.values()
        }
        comparison = {
            "image_id": image_id,
            "captions_all_identical": len(set(captions)) == 1,
            "token_ids_all_identical": len(set(token_sequences)) == 1,
            "continuous_probe_values_changed": len(probe_signatures) > 1,
            "candidates": rows,
        }
        comparisons.append(comparison)

        image = data.load_image(image_id)
        print(
            f"IMAGE {image_index}/{len(VAL_BENCHMARK_IDS)}:",
            image_id,
        )
        display(image)
        for candidate_name, _ in candidate_configs:
            probe = rows[candidate_name]["probe"]
            print(
                f"  {candidate_name}:",
                rows[candidate_name]["caption"],
                "| loss_total=",
                f"{probe['loss_total']:.6f}",
                "| p_top=",
                f"{probe['p_fused_top_probability']:.6f}",
                "| cache_max_err=",
                f"{probe['cache_max_abs_error']:.3e}",
            )
        print(
            "  identical caption/token IDs:",
            comparison["captions_all_identical"],
            comparison["token_ids_all_identical"],
            "| continuous probe changed:",
            comparison["continuous_probe_values_changed"],
        )
        del image

    print(
        "All candidate predictions completed without references. "
        "VAL evaluation starts now."
    )
    candidate_metrics = {}
    for candidate_name, candidate_config in candidate_configs:
        evaluator = Flickr8kEvaluator(candidate_config)
        try:
            metrics = evaluator.evaluate(
                data=data,
                image_ids=VAL_BENCHMARK_IDS,
                predictions=(
                    candidate_stores[candidate_name].predictions
                ),
                split="val",
            )
        except Exception as exc:
            metrics = {
                "num_images": len(VAL_BENCHMARK_IDS),
                "evaluation_split": "val",
                "run_mode": candidate_config.run_mode,
                "config_hash": candidate_config.config_hash,
                "git_commit": candidate_config.git_commit,
                "fatal_evaluation_error": (
                    f"{type(exc).__name__}: {exc}"
                ),
                "predictions_preserved": True,
            }
        candidate_stores[candidate_name].save_metrics(metrics)
        candidate_metrics[candidate_name] = metrics
        print(
            candidate_name,
            "CIDEr=",
            metrics.get("CIDEr"),
            "METEOR=",
            metrics.get("METEOR"),
            "ROUGE-L=",
            metrics.get("ROUGE-L"),
        )

    cider_rows = [
        (
            candidate_name,
            float(candidate_metrics[candidate_name]["CIDEr"]),
        )
        for candidate_name, _ in candidate_configs
        if isinstance(
            candidate_metrics[candidate_name].get("CIDEr"),
            (int, float),
        )
    ]
    winner_by_cider = (
        max(cider_rows, key=lambda item: item[1])[0]
        if cider_rows
        else None
    )
    cider_ranking = [
        {"candidate": name, "CIDEr": score, "rank": rank}
        for rank, (name, score) in enumerate(
            sorted(cider_rows, key=lambda item: item[1], reverse=True),
            start=1,
        )
    ]
    recommended_for_confirmation = [
        row["candidate"] for row in cider_ranking[:2]
    ]

    baseline_name = "A_baseline"
    comparison_rows = []
    for image_id in VAL_BENCHMARK_IDS:
        for candidate_name, candidate_config in candidate_configs:
            prediction = candidate_stores[candidate_name].predictions[image_id]
            probe = trace_probes[candidate_name][image_id]
            comparison_rows.append({
                "image_id": image_id,
                "candidate": candidate_name,
                "case_reason": candidate_config.val_tune_case_reason,
                "config_hash": candidate_config.config_hash,
                "fluency_weight": float(candidate_config.fluency_weight),
                "fusion_factor": float(candidate_config.fusion_factor),
                "clip_temperature": float(
                    candidate_config.clip_temperature
                ),
                "caption": prediction["caption"],
                "generated_token_ids": json.dumps(
                    prediction["generated_token_ids"]
                ),
                "stop_reason": prediction["stop_reason"],
                "num_generated_tokens": prediction[
                    "num_generated_tokens"
                ],
                "generation_time_sec": prediction[
                    "generation_time_sec"
                ],
                "final_clip_similarity": prediction[
                    "final_clip_similarity"
                ],
                "cache_max_abs_error": probe["cache_max_abs_error"],
                "loss_clip": probe["loss_clip"],
                "loss_fluency": probe["loss_fluency"],
                "weighted_fluency_loss": probe[
                    "weighted_fluency_loss"
                ],
                "loss_total": probe["loss_total"],
                "delta_global_norm": probe["delta_global_norm"],
                "p_fused_entropy": probe["p_fused_entropy"],
                "p_fused_top_token_id": probe[
                    "p_fused_top_token_id"
                ],
                "p_fused_top_token_text": probe[
                    "p_fused_top_token_text"
                ],
                "p_fused_top_probability": probe[
                    "p_fused_top_probability"
                ],
            })

    comparison_csv_path = RUN_DIR / "val_tune_comparison.csv"
    comparison_csv_part = comparison_csv_path.with_suffix(".csv.part")
    pd.DataFrame(comparison_rows).to_csv(
        comparison_csv_part,
        index=False,
    )
    os.replace(comparison_csv_part, comparison_csv_path)
    store.save_artifact_json(
        "val_tune_comparison.json",
        comparison_rows,
    )

    metric_names = ("CIDEr", "METEOR", "ROUGE-L", "BLEU-1", "BLEU-4")
    baseline_metrics = candidate_metrics.get(baseline_name, {})
    baseline_predictions = candidate_stores[baseline_name].predictions
    candidate_explanations = []
    for candidate_name, candidate_config in candidate_configs:
        predictions = candidate_stores[candidate_name].predictions
        changed_caption_ids = [
            image_id
            for image_id in VAL_BENCHMARK_IDS
            if predictions[image_id]["caption"]
            != baseline_predictions[image_id]["caption"]
        ]
        changed_token_ids = [
            image_id
            for image_id in VAL_BENCHMARK_IDS
            if predictions[image_id]["generated_token_ids"]
            != baseline_predictions[image_id]["generated_token_ids"]
        ]
        metric_deltas = {}
        for metric_name in metric_names:
            candidate_value = candidate_metrics[candidate_name].get(metric_name)
            baseline_value = baseline_metrics.get(metric_name)
            if isinstance(candidate_value, (int, float)) and isinstance(
                baseline_value, (int, float)
            ):
                metric_deltas[metric_name] = float(
                    candidate_value - baseline_value
                )
        candidate_explanations.append({
            "candidate": candidate_name,
            "case_reason": candidate_config.val_tune_case_reason,
            "changed_caption_count_vs_baseline": len(
                changed_caption_ids
            ),
            "changed_caption_image_ids": changed_caption_ids,
            "changed_token_count_vs_baseline": len(changed_token_ids),
            "changed_token_image_ids": changed_token_ids,
            "metric_deltas_vs_baseline": metric_deltas,
            "mean_first_step_probe": {
                "loss_clip": float(np.mean([
                    trace_probes[candidate_name][image_id]["loss_clip"]
                    for image_id in VAL_BENCHMARK_IDS
                ])),
                "loss_fluency": float(np.mean([
                    trace_probes[candidate_name][image_id][
                        "loss_fluency"
                    ]
                    for image_id in VAL_BENCHMARK_IDS
                ])),
                "weighted_fluency_loss": float(np.mean([
                    trace_probes[candidate_name][image_id][
                        "weighted_fluency_loss"
                    ]
                    for image_id in VAL_BENCHMARK_IDS
                ])),
                "p_fused_entropy": float(np.mean([
                    trace_probes[candidate_name][image_id][
                        "p_fused_entropy"
                    ]
                    for image_id in VAL_BENCHMARK_IDS
                ])),
                "p_fused_top_probability": float(np.mean([
                    trace_probes[candidate_name][image_id][
                        "p_fused_top_probability"
                    ]
                    for image_id in VAL_BENCHMARK_IDS
                ])),
            },
            "mean_final_clip_similarity": float(np.mean([
                predictions[image_id]["final_clip_similarity"]
                for image_id in VAL_BENCHMARK_IDS
            ])),
            "mean_generation_time_sec": float(np.mean([
                predictions[image_id]["generation_time_sec"]
                for image_id in VAL_BENCHMARK_IDS
            ])),
            "observed_effect": (
                "baseline reference"
                if candidate_name == baseline_name
                else (
                    f"Changed {len(changed_caption_ids)}/"
                    f"{len(VAL_BENCHMARK_IDS)} captions versus baseline; "
                    "inspect per-image traces before causal interpretation."
                )
            ),
        })

    selection_report = {
        "selection_scope": "diagnostic fixed VAL subset only",
        "primary_metric": "CIDEr",
        "ranking_by_cider": cider_ranking,
        "provisional_winner": winner_by_cider,
        "recommended_for_larger_val_confirmation": (
            recommended_for_confirmation
        ),
        "candidate_explanations": candidate_explanations,
        "interpretation_rules": [
            "Do not compare loss_total numerically across different "
            "fluency_weight values because the objective itself changes.",
            "Use token/probability traces to explain mechanisms; use VAL "
            "caption metrics and manual grounding review to select.",
            "CLIPScore is supplementary because the same CLIP backbone "
            "guides ZeroCap generation.",
            "Confirm the top two cases on a larger fixed VAL set before "
            "locking one config; never tune on TEST.",
        ],
    }
    store.save_artifact_json("selection_report.json", selection_report)
    all_captions_identical = all(
        row["captions_all_identical"] for row in comparisons
    )
    all_token_ids_identical = all(
        row["token_ids_all_identical"] for row in comparisons
    )
    any_continuous_probe_changed = any(
        row["continuous_probe_values_changed"]
        for row in comparisons
    )
    if all_captions_identical and any_continuous_probe_changed:
        sensitivity_interpretation = (
            "config_applied_but_discrete_decoding_unchanged"
        )
    elif all_captions_identical:
        sensitivity_interpretation = (
            "no_observed_change_even_in_continuous_probe"
        )
    else:
        sensitivity_interpretation = "grid_changed_generated_output"

    VAL_TUNE_SUMMARY = {
        "protocol": {
            "purpose": "VAL-only inference hyperparameter sensitivity",
            "not_model_training": True,
            "benchmark_unchanged": True,
            "baseline_fidelity": {
                "prepend_gpt2_bos": True,
                "remove_leading_bos_before_clip_text": True,
                "final_rerank_text": "caption_only",
            },
            "image_ids": list(VAL_BENCHMARK_IDS),
            "references_loaded_after_all_predictions": True,
            "zero_delta_cache_equivalence_checked": True,
            "selection_warning": (
                "Five VAL images are diagnostic only; do not promote a "
                "candidate to final TEST without a larger fixed VAL confirmation."
            ),
        },
        "suite_run_dir": str(RUN_DIR),
        "git_commit": config.git_commit,
        "algorithm_revision": config.algorithm_revision,
        "candidates": [
            {
                "name": candidate_name,
                "config_hash": candidate_config.config_hash,
                "run_dir": candidate_config.run_dir,
                "case_reason": candidate_config.val_tune_case_reason,
                "fluency_weight": float(
                    candidate_config.fluency_weight
                ),
                "fusion_factor": float(
                    candidate_config.fusion_factor
                ),
                "clip_temperature": float(
                    candidate_config.clip_temperature
                ),
                "prepend_bos_token": bool(
                    candidate_config.prepend_bos_token
                ),
                "final_rerank_text_mode": (
                    candidate_config.final_rerank_text_mode
                ),
                "effective_config": effective_configs[candidate_name],
                "metrics": candidate_metrics[candidate_name],
            }
            for candidate_name, candidate_config in candidate_configs
        ],
        "per_image_comparisons": comparisons,
        "sensitivity_diagnosis": {
            "all_captions_identical": all_captions_identical,
            "all_token_ids_identical": all_token_ids_identical,
            "any_continuous_probe_changed": (
                any_continuous_probe_changed
            ),
            "interpretation": sensitivity_interpretation,
        },
        "winner_by_cider_diagnostic_only": winner_by_cider,
        "recommended_for_larger_val_confirmation": (
            recommended_for_confirmation
        ),
        "selection_report": selection_report,
        "comparison_csv": str(comparison_csv_path),
    }
    store.save_artifact_json(
        "val_tune_summary.json",
        VAL_TUNE_SUMMARY,
    )
    print(
        "VAL-tune summary saved:",
        RUN_DIR / "val_tune_summary.json",
    )
    print(
        "Sensitivity interpretation:",
        sensitivity_interpretation,
    )
    print("Comparison CSV saved:", comparison_csv_path)
    print("Selection report saved:", RUN_DIR / "selection_report.json")
    print(
        "Diagnostic-only CIDEr winner:",
        winner_by_cider,
    )
    print(
        "Top cases for larger fixed-VAL confirmation:",
        recommended_for_confirmation,
    )
    print(
        "VAL_TUNE PIPELINE: PASS — model weights remained frozen; "
        "benchmark and TEST were not run."
    )
else:
    print("VAL-tune cell skipped for RUN_MODE =", config.run_mode)

## 19. Output handoff

In [ ]:
if MOUNT_DRIVE_FOR_OUTPUTS:
    print("Outputs are persisted in Google Drive:", RUN_DIR)
else:
    archive_base = str(RUN_DIR.parent / RUN_DIR.name)
    archive_path = shutil.make_archive(
        archive_base,
        "zip",
        root_dir=RUN_DIR,
    )
    print("Created output archive:", archive_path)
    from google.colab import files

    files.download(archive_path)

if config.save_diagnostics:
    print("Diagnostic JSON directory:", RUN_DIR / "diagnostics")
    print("When reporting a problem, share the whole run directory/ZIP.")

## 20. Migration mapping — do not migrate yet

Sau khi notebook prototype PASS trên Colab T4, logic đã kiểm chứng sẽ được trích xuất nguyên trạng, chỉ sửa imports và tách module:

- ZeroCapRunConfig → src/config/zerocap_config.py
- GenerationResult → src/zerocap/types.py
- ZeroCapModels → src/zerocap/model_loader.py
- ImageEncoder → src/zerocap/image_encoder.py
- CLIPGuidance → src/zerocap/clip_guidance.py
- ContextOptimizer → src/zerocap/context_optimizer.py
- ZeroCapDecoder → src/zerocap/decoding.py
- ZeroCapGenerator → src/zerocap/generator.py
- ZeroCapCaptioner → src/zerocap/captioner.py

Regression sau migration: Restart Colab → chạy cùng fixed VAL IDs → so caption, loss diagnostics và runtime với prototype → chỉ chạy final TEST khi bản module PASS.